# Metin Temsillerini Karşılaştırmak: Sayımlar ve Embedding'ler — eşlik eden notebook

*Uygulamalı Doğal Dil İşleme — Token'lardan Ajanlara* kitabının 4. bölümü.
Önce bölümü okuyun: **[Metin Temsillerini Karşılaştırmak: Sayımlar ve Embedding'ler](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/04-karsilastirma.html)**.

Bölüm, 2. bölümün sayım yöntemlerini ve 3. bölümün embedding yöntemlerini tek bir görevde;
aynı bölmeyle, aynı sınıflandırıcıyla ve aynı metrikle karşılaştırıyor. Ardından ortaya çıkan
tablonun neyi gösterdiğini, neyi göstermediğini soruyor. Bu notebook o karşılaştırmayı
bölümün sırasıyla çalıştırıyor:

1. **Veri ve bölme**: 235.165 Türkçe ürün yorumu, tekilleştirmeden önceki kontroller ve
   her bölmedeki satırların parmak izi.
2. **İlk skorun sakladığı sınıf hatası**: hiç "olumsuz" demeyen bir modelin aldığı skor.
3. **Türkçeyi token'lara ayırmak**: noktalı ve noktasız i, ve `str.lower()` çağrısının
   onlara yaptığı.
4. **Birinci raunt**: bag-of-words (kelime torbası), TF-IDF, kelime bigram'ları ve karakter
   n-gram'ları.
5. **İkinci raunt**: aynı 40.000 yorum üzerinde eğitilmiş, ortalaması alınmış word2vec,
   ortalaması alınmış fastText ve doc2vec.
6. **Aralıklar**: komşu satır çiftlerinin her biri için eşleştirilmiş bootstrap.
7. **Sözlük dışı kelimeler**, hata örnekleri ve bir yorum ile olumsuzlanmış hâli arasındaki
   kosinüs.
8. **Kesişim noktası**: aynı karşılaştırma, 100 ile 40.000 arasında etiketle.
9. **Maliyet, işe yaramayan bir melez ve kelime sırası.**

**Sayılar nasıl okunur.** Aşağıda basılan her şey *bu koşunun* ürettiği şey. Bölüm aynı
niceliği basıyorsa hücre kitabın değerini yanına yazıyor ve ikisinin kitabın bastığı
hassasiyette tutup tutmadığını söylüyor. Seyrek satırlar (sayımlar, TF-IDF) birebir yeniden
üretiliyor. Yoğun satırlar (word2vec, fastText, doc2vec) için bölümün bir uyarısı var:
ikinci raunt kısmı bu satırların son hanesine kadar yeniden üretilemediğini söylüyor ve
kendi ikinci koşusunun değerlerini veriyor. O satırlarda hücreler eşleşme yerine kitabın
değerine olan uzaklığı bildiriyor; üçünden hangisinin koşudan koşuya gerçekten oynadığını da
altıncı kısım (6) gösteriyor. Süreler makinenize bağlı; onları büyüklük mertebesi olarak okuyun.

**Dil hakkında bir not.** Anlatım ve kod yorumları Türkçe; kodun kendisi ve *basılan*
etiketler İngilizce (`same as the book`, `DIFFERS`, `not comparable` gibi). Sebebi, Türkçe
bölümün kendi `text` bloklarının da İngilizce olması: burada bastığınız tabloyu kitaptaki
tabloyla yan yana okuyabilirsiniz.

**Süre ve indirmeler.** Bir dizüstü CPU'sunda (Apple M serisi) tam koşu yaklaşık 7 dakika.
`ANLP_SKIP_HEAVY=1` ile yaklaşık 3 dakika. Tek indirme veri kümesi (24 MB); bu notebook'un
yanındaki `data/cache/` altında önbelleğe alınıyor.

**`ANLP_SKIP_HEAVY=1`** (CI bunu ayarlıyor) üç şeyi atlamak yerine kısaltıyor: bootstrap'ler
10.000 yerine 1.000 yeniden örneklem çekiyor, tohum yayılımı hücresi beş yerine bir tohum
eğitiyor ve melezin doğrulama ızgarası bölümdeki aramanın seçtiği tek noktaya iniyor. Bu
hücreler bunu söylüyor ve kendilerini kitapla karşılaştırmıyor.

## 1. Kurulum

İlk hücre bu koşunun kullandığı yorumlayıcıyı ve paket sürümlerini kaydediyor. Bölümün
durum notu `scikit-learn` 1.9 ve `gensim` 4.4 diyor. Sizin sayılarınız bu notebook'ta
kayıtlı olanlardan farklıysa önce bu hücreyi karşılaştırın.

In [1]:
import collections
import hashlib
import os
import platform
import re
import time
import unicodedata
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import scipy.sparse as sp
from scipy.stats import binomtest
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize

SEED = 42
SKIP_HEAVY = os.environ.get("ANLP_SKIP_HEAVY") == "1"
N_BOOT = 1_000 if SKIP_HEAVY else 10_000

print(f"Python        {platform.python_version()}  ({platform.system()} {platform.machine()})")
for pkg in ["numpy", "pandas", "scipy", "scikit-learn", "gensim", "pyarrow", "requests"]:
    print(f"{pkg:<13} {version(pkg)}")
print("\nANLP_SKIP_HEAVY", "= 1: some cells run a shortened version" if SKIP_HEAVY
      else "is not set: every cell runs in full")

Python        3.12.13  (Darwin arm64)
numpy         2.5.2
pandas        3.0.5
scipy         1.18.1
scikit-learn  1.9.0
gensim        4.4.0
pyarrow       25.0.1
requests      2.34.2

ANLP_SKIP_HEAVY is not set: every cell runs in full


Baştan sona kullanılan üç yardımcı.

- `vs_book` bu koşunun sayısını kitabınkinin yanına basıyor ve ikisini kitabın bastığı
  hassasiyette karşılaştırıyor.
- `vs_book_near` yoğun satırlar için. Kitabın değerinden farkı basıyor ve farkın belirtilen
  toleransın içinde olup olmadığını söylüyor.
- `scores` doğruluğu (accuracy), macro-F1'i ve olumsuz sınıfın F1'ini yüzde olarak döndürüyor. Macro-F1
  çağrısı bölümünki: `f1_score(..., average="macro")`.

In [2]:
def vs_book(label, ours, book, fmt="{}", note=None):
    """Bu koşunun değeri kitabınkinin yanında; kitabın bastığı hassasiyette karşılaştırılır."""
    ours_s, book_s = fmt.format(ours), fmt.format(book)
    if note:
        verdict = f"not comparable: {note}"
    else:
        verdict = "same as the book" if ours_s == book_s else "DIFFERS from the book"
    print(f"{label:<38} this run {ours_s:>10}   book {book_s:>10}   {verdict}")


def vs_book_near(label, ours, book, tol, fmt="{:.2f}", note=None):
    """Bit düzeyinde yeniden üretilemeyen satırlar için: kitabın değerine olan uzaklık."""
    if note:
        verdict = f"not comparable: {note}"
    else:
        verdict = (f"{ours - book:+.2f} from the book, "
                   + ("inside" if abs(ours - book) <= tol else "OUTSIDE") + f" ±{tol}")
    print(f"{label:<38} this run {fmt.format(ours):>10}   book {fmt.format(book):>10}   {verdict}")


def scores(y_true, y_pred):
    """Doğruluk (accuracy), macro-F1 ve olumsuz sınıfın F1'i, yüzde olarak."""
    return (100 * accuracy_score(y_true, y_pred),
            100 * f1_score(y_true, y_pred, average="macro", zero_division=0),
            100 * f1_score(y_true, y_pred, pos_label=0, zero_division=0))

## 2. Problem: veri ve bölme

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/04-karsilastirma.html#problem)

Derlem, Hugging Face'teki `fthbrmnby/turkish_product_reviews`: olumlu ya da olumsuz diye
etiketlenmiş Türkçe ürün yorumları. Fatih Barmanbay derlemiş, **CC BY-SA 4.0** ile
dağıtılıyor. Bölüm, veri kümesi kartının söylemediklerini de not ediyor: yorumların hangi
siteden geldiği, ne zaman toplandığı ve olumlu/olumsuz etiketinin nasıl türetildiği.

Yükleme kodu bölümün. Tek değişiklik yerel bir önbellek; notebook yeniden çalıştırıldığında
dosya bir daha indirilmiyor. Bölüm tekilleştirmeden *önce* üç kontrol yapıyor, çünkü
`drop_duplicates` iki etiketi birden taşıyan bir metne ne olacağına sessizce karar veriyor:
ilk satırı tutuyor.

**Neye bakacaksınız:** boşlukları kırpılınca 21 metin boş kalıyor ve 77 metin iki etiketle
birden geçiyor.

In [3]:
URL = ("https://huggingface.co/datasets/fthbrmnby/turkish_product_reviews/"
       "resolve/main/data/train-00000-of-00001.parquet")
CACHE = Path("data/cache/turkish_product_reviews.parquet")

if not CACHE.exists():
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    response = requests.get(URL, timeout=300)
    response.raise_for_status()
    CACHE.write_bytes(response.content)
print(f"parquet file  {CACHE.stat().st_size:,} bytes, "
      f"sha256 {hashlib.sha256(CACHE.read_bytes()).hexdigest()[:12]}\n")

# bölümün yükleme kodu
df = pd.read_parquet(CACHE).rename(
    columns={"sentence": "text", "sentiment": "label"})
n_raw = len(df)
n_missing = int(df.isna().any(axis=1).sum())
df["text"] = df["text"].astype(str).str.strip()
labels_per_text = df.groupby("text")["label"].nunique()
n_empty = int(df["text"].eq("").sum())
n_both = int(labels_per_text.gt(1).sum())
df = df.drop_duplicates(subset="text")

print("### checks before de-duplication")
vs_book("missing text or label", n_missing, 0, "{:,}")
vs_book("empty texts", n_empty, 21, "{:,}")
vs_book("texts with both labels", n_both, 77, "{:,}")
print("\n### the raw data")
vs_book("rows as downloaded", n_raw, 235_165, "{:,}")
vs_book("after de-duplication", len(df), 233_508, "{:,}")
vs_book("exact duplicates removed", n_raw - len(df), 1_657, "{:,}")
vs_book("label 1 (positive)", int((df["label"] == 1).sum()), 218_747, "{:,}")
vs_book("label 0 (negative)", int((df["label"] == 0).sum()), 14_761, "{:,}")
vs_book("positive share, %", 100 * (df["label"] == 1).mean(), 93.7, "{:.1f}")

parquet file  24,354,762 bytes, sha256 a5ea59cd4f78

### checks before de-duplication
missing text or label                  this run          0   book          0   same as the book
empty texts                            this run         21   book         21   same as the book
texts with both labels                 this run         77   book         77   same as the book

### the raw data
rows as downloaded                     this run    235,165   book    235,165   same as the book
after de-duplication                   this run    233,508   book    233,508   same as the book
exact duplicates removed               this run      1,657   book      1,657   same as the book
label 1 (positive)                     this run    218,747   book    218,747   same as the book
label 0 (negative)                     this run     14,761   book     14,761   same as the book
positive share, %                      this run       93.7   book       93.7   same as the book


### Bölme ve parmak izi

Önce 50.000 yorumluk tabakalı bir örneklem, sonra tabakalı bir 80/20 bölme; ikisi de 42
tohumuyla. "Tabakalı, tohum 42" tek bir işlemi değil, bir işlemler ailesini adlandırıyor;
bu yüzden bölüm iki çağrıyı ve her parçadaki satır indekslerinin parmak izini basıyor.
Parmak izleriniz tutuyorsa elinizde satırı satırına aynı 40.000 eğitim ve 10.000 test
yorumu var.

In [4]:
# bölümün bölmesi
sub, _ = train_test_split(df, train_size=50_000,
                          stratify=df["label"], random_state=42)
train, test = train_test_split(sub, test_size=0.2,
                               stratify=sub["label"], random_state=42)

def fingerprint(frame):
    rows = ",".join(map(str, sorted(frame.index)))
    return hashlib.sha256(rows.encode()).hexdigest()[:12]

print("### the split (stratified, seed 42)")
for name, part in [("train", train), ("test", test)]:
    pos = int(part["label"].sum())
    print(f"{name:<6} {len(part):>7,}   {pos:>7,} positive   "
          f"{len(part) - pos:>6,} negative   rows {fingerprint(part)}")

y_train, y_test = train["label"].to_numpy(), test["label"].to_numpy()
print()
vs_book("train fingerprint", fingerprint(train), "8205c2be5ebf")
vs_book("test fingerprint", fingerprint(test), "634cdb3bb002")
vs_book("negatives in train", int((y_train == 0).sum()), 2_529, "{:,}")
vs_book("negatives in test", int((y_test == 0).sum()), 632, "{:,}")

### the split (stratified, seed 42)
train   40,000    37,471 positive    2,529 negative   rows 8205c2be5ebf
test    10,000     9,368 positive      632 negative   rows 634cdb3bb002

train fingerprint                      this run 8205c2be5ebf   book 8205c2be5ebf   same as the book
test fingerprint                       this run 634cdb3bb002   book 634cdb3bb002   same as the book
negatives in train                     this run      2,529   book      2,529   same as the book
negatives in test                      this run        632   book        632   same as the book


## 3. İlk skorun sakladığı sınıf hatası

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/04-karsilastirma.html#hesaplayacağınız-ilk-sayının-içindeki-tuzak)

Her yoruma "olumlu" deyin ve metne hiç bakmayın. Bölüm bu modelin karışıklık matrisini
bütünüyle basıyor, çünkü doğruluk değeri bir sonuç gibi görünüyor.

**Neye bakacaksınız:** doğruluk, test kümesindeki olumlu yorumların payına eşit; olumsuz
sınıfın F1'i ise sıfır: model 632 olumsuz yorumun hiçbirini bulmuyor. Macro-F1 iki sınıfın
F1 skorlarının ağırlıksız ortalaması, dolayısıyla yarıya iniyor.

In [5]:
majority_pred = np.ones_like(y_test)
cm = confusion_matrix(y_test, majority_pred, labels=[0, 1])
print("### majority baseline, in full")
print("                 predicted neg   predicted pos   total")
for i, name in enumerate(["actual neg", "actual pos"]):
    print(f"   {name}   {cm[i, 0]:>8,}   {cm[i, 1]:>13,}   {cm[i].sum():>10,}")
f1_neg = f1_score(y_test, majority_pred, pos_label=0, zero_division=0)
f1_pos = f1_score(y_test, majority_pred, pos_label=1)
print()
majority_acc, majority_macro, _ = scores(y_test, majority_pred)
vs_book("accuracy, %", majority_acc, 93.68, "{:.2f}")
vs_book("F1 (neg)", 100 * f1_neg, 0.00, "{:.2f}")
vs_book("F1 (pos)", 100 * f1_pos, 96.74, "{:.2f}")
vs_book("macro-F1", majority_macro, 48.37, "{:.2f}")

### majority baseline, in full
                 predicted neg   predicted pos   total
   actual neg          0             632          632
   actual pos          0           9,368        9,368

accuracy, %                            this run      93.68   book      93.68   same as the book
F1 (neg)                               this run       0.00   book       0.00   same as the book
F1 (pos)                               this run      96.74   book      96.74   same as the book
macro-F1                               this run      48.37   book      48.37   same as the book


**Nasıl okunur.** Model hiç olumsuz demediği için olumsuz sınıfın precision'ı 0/0. Bölüm
bunu *tanımsız* diye bildiriyor ve F1'i tanımlı olan sayım biçimiyle veriyor:
$2\,TP / (2\,TP + FP + FN) = 0 / 632$. `scikit-learn`'e `zero_division=0` vermek yalnızca
tanımsız precision uyarısını susturmak için gerekiyor.

Aşağıdaki her tablonun her satırı sıfıra karşı değil, **48,37 macro-F1**'e karşı
okunmalı.

## 4. Türkçeyi token'lara ayırmak

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/04-karsilastirma.html#türkçeyi-tokenlara-ayırmak-ve-oradaki-tuzak)

İngilizcenin iki harfinin olduğu yerde Türkçenin dört harfi var: noktalı `İ`/`i` ve
noktasız `I`/`ı`. Python'ın `str.lower()` çağrısı Unicode'un varsayılan kuralını izliyor;
bu kural Türkçenin kuralı değil. `I` harfini `i` yapıyor (iki farklı harfi birleştiriyor),
`İ` harfini ise ardından birleştirici nokta (U+0307) gelen bir `i` yapıyor.

**Neye bakacaksınız:** `"İYİ".lower()` sonucunun uzunluğu. Üç harf giriyor, beş kod noktası
çıkıyor.

In [6]:
TR_LOWER = str.maketrans({"I": "ı", "İ": "i"})
def tr_lower(s):
    return s.translate(TR_LOWER).lower()

print("### the same string, two lowercasing rules")
print(f"{'':<12}{'str.lower()':<26}{'tr_lower()'}")
for s in ["IYI", "İYİ"]:
    a, b = s.lower(), tr_lower(s)
    print(f'"{s}"       {a!r:<8} {len(a)} code points     {b!r:<8} {len(b)} code points')
print("\ncode points of 'İYİ'.lower():",
      " ".join(f"U+{ord(c):04X}" for c in "İYİ".lower()))

### the same string, two lowercasing rules
            str.lower()               tr_lower()
"IYI"       'iyi'    3 code points     'ıyı'    3 code points
"İYİ"       'i̇yi̇'  5 code points     'iyi'    3 code points

code points of 'İYİ'.lower(): U+0069 U+0307 U+0079 U+0069 U+0307


Bölümün tokenizer'ı: Türkçe kuralıyla küçük harfe çevir, sonra Türkçe küçük harf dizilerini
tut. Rakamlar, noktalama ve emojiler atılıyor; birleştirici nokta da atılıyor ve onu içeren
bir kelime bu yüzden ikiye bölünüyor.

Aşağıdaki basım bizim; istatistikler bölümün 40.000 eğitim yorumu için verdiği tablo.

In [7]:
TOKEN_RE = re.compile(r"[a-zçğıöşü]+")
def tokenize(s):
    return TOKEN_RE.findall(tr_lower(s))

train_tok = [tokenize(s) for s in train["text"]]
test_tok = [tokenize(s) for s in test["text"]]
counts = collections.Counter(w for doc in train_tok for w in doc)
lengths = np.array([len(doc) for doc in train_tok])
hapax = sum(1 for c in counts.values() if c == 1)

print("### the training corpus, 40,000 reviews")
vs_book("tokens", sum(counts.values()), 903_190, "{:,}")
vs_book("distinct word forms", len(counts), 74_317, "{:,}")
vs_book("hapax (seen exactly once)", hapax, 43_579, "{:,}")
vs_book("hapax share of forms, %", 100 * hapax / len(counts), 58.6, "{:.1f}")
vs_book("mean review length, tokens", lengths.mean(), 22.6, "{:.1f}")
vs_book("median", int(np.median(lengths)), 19)
vs_book("95th percentile", int(np.percentile(lengths, 95)), 56)
vs_book("longest", int(lengths.max()), 296)

### the training corpus, 40,000 reviews
tokens                                 this run    903,190   book    903,190   same as the book
distinct word forms                    this run     74,317   book     74,317   same as the book
hapax (seen exactly once)              this run     43,579   book     43,579   same as the book
hapax share of forms, %                this run       58.6   book       58.6   same as the book
mean review length, tokens             this run       22.6   book       22.6   same as the book
median                                 this run         19   book         19   same as the book
95th percentile                        this run         56   book         56   same as the book
longest                                this run        296   book        296   same as the book


Kelime biçimlerinin yarısından fazlası tam bir kez görülüyor. Bir nedeni biçimbilim: tek
bir Türkçe gövde yüzlerce yüzey biçimiyle karşımıza çıkıyor. Bölüm, beş karakterlik aynı
ön eki paylaşan farklı biçimleri sayıyor.

**Neye bakacaksınız:** yalnızca `kulla` (*kullanmak* fiilinin gövdesi), kelime düzeyindeki
bir sözlüğün 895 sütununu oluşturuyor.

In [8]:
by_prefix = collections.defaultdict(list)
for form, count in counts.most_common():          # en sık biçim önce
    if len(form) >= 5:
        by_prefix[form[:5]].append(form)

print("### distinct word forms sharing a five-character prefix")
for prefix, book in [("kulla", 895), ("bekle", 285), ("çalış", 247), ("çıkar", 209), ("düşün", 195)]:
    forms = by_prefix[prefix]
    print(f"{prefix}   {', '.join(forms[:5])}, ...")
    vs_book(f"  forms starting with {prefix}", len(forms), book)

### distinct word forms sharing a five-character prefix
kulla   kullanışlı, kullanıyorum, kullandım, kullanımı, kullanım, ...
  forms starting with kulla            this run        895   book        895   same as the book
bekle   beklediğimden, beklediğim, beklemiyordum, beklemeyin, beklentimin, ...
  forms starting with bekle            this run        285   book        285   same as the book
çalış   çalışıyor, çalışma, çalışan, çalışması, çalışırken, ...
  forms starting with çalış            this run        247   book        247   same as the book
çıkar   çıkarıyor, çıkar, çıkartıyor, çıkarmıyor, çıkardı, ...
  forms starting with çıkar            this run        209   book        209   same as the book
düşün   düşünüyorum, düşünmeden, düşünenlere, düşünen, düşünmeyin, ...
  forms starting with düşün            this run        195   book        195   same as the book


## 5. Birinci raunt: 2. bölümün yöntemleri

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/04-karsilastirma.html#raunt-2.-bölümün-yöntemleri)

Dört sayma biçimi, tek sınıflandırıcı. Vektörleştiriciler bölümün; `scikit-learn`'ün kendi
başına `str.lower()` çağırmasını engelleyen iki argüman da dâhil. Her vektörleştirici
yalnızca eğitim yorumlarında eğitiliyor, sonra test yorumlarına uygulanıyor.

Sınıflandırıcı bölümün her satırında aynı: `class_weight="balanced"` ile lojistik
regresyon. Bu ayar 2.529 olumsuz eğitim yorumuna 37.471 olumlu yorum kadar ağırlık veriyor.

`fit` sütunu, eğitim matrisini kurup sınıflandırıcıyı eğitmenin bu makinedeki süresi.

**Neye bakacaksınız:** doğruluk ve macro-F1 satırları farklı sıralıyor. Dört modelin üçünün
doğruluğu çoğunluk kuralından (baseline, yani karşılaştırma modeli) *düşük*, macro-F1'i
ise çok daha yüksek.

In [9]:
PAT = r"[a-zçğıöşü]+"
# lowercase=False + preprocessor=tr_lower: aksi hâlde scikit-learn Python'ın
# str.lower()'ını çağırır ve yukarıda uyardığımız hatayı biz yaparız
LOC = dict(preprocessor=tr_lower, lowercase=False)

def make_schemes():
    return {
      "bag of words":    CountVectorizer(min_df=2, token_pattern=PAT, **LOC),
      "tf-idf":          TfidfVectorizer(min_df=2, sublinear_tf=True, token_pattern=PAT, **LOC),
      "tf-idf 1-2 gram": TfidfVectorizer(min_df=2, sublinear_tf=True,
                                         ngram_range=(1, 2), token_pattern=PAT, **LOC),
        # ayrı öznitelik çıkarma yolu: token_pattern kullanılmaz, min_df de farklı
      "tf-idf char_wb 3-5": TfidfVectorizer(min_df=3, sublinear_tf=True,
                                            analyzer="char_wb", ngram_range=(3, 5), **LOC),
    }

def new_clf(C=1.0, max_iter=2000):
    return LogisticRegression(C=C, max_iter=max_iter, class_weight="balanced", random_state=42)

BOOK_ROUND1 = {"bag of words": (92.14, 73.51, 51.30, 30_195), "tf-idf": (90.07, 71.95, 49.41, 30_195),
               "tf-idf 1-2 gram": (92.57, 76.21, 56.47, 101_659), "tf-idf char_wb 3-5": (88.63, 70.39, 47.14, 94_939)}

schemes = make_schemes()
X_train, X_test, pred, cost = {}, {}, {}, {}
print("### round 1: counting")
print(f"{'':<22}accuracy   macro-F1   F1(neg)      dims      fit")
print(f"{'majority baseline':<22}{majority_acc:8.2f} {majority_macro:10.2f} {0:9.2f}         —        —")
for name, vec in schemes.items():
    start = time.perf_counter()
    X_train[name] = vec.fit_transform(train["text"])
    t_build = time.perf_counter() - start
    start = time.perf_counter()
    X_test[name] = vec.transform(test["text"])
    t_test = time.perf_counter() - start
    start = time.perf_counter()
    model = new_clf().fit(X_train[name], y_train)
    t_fit = time.perf_counter() - start
    pred[name] = model.predict(X_test[name])
    cost[name] = (t_build, t_test, X_train[name])
    acc, macro, f1n = scores(y_test, pred[name])
    print(f"{name:<22}{acc:8.2f} {macro:10.2f} {f1n:9.2f} {X_train[name].shape[1]:>9,} {t_build + t_fit:7.1f} s")

print()
for name, (b_acc, b_macro, b_f1n, b_dims) in BOOK_ROUND1.items():
    acc, macro, f1n = scores(y_test, pred[name])
    vs_book(f"{name}: accuracy", acc, b_acc, "{:.2f}")
    vs_book(f"{name}: macro-F1", macro, b_macro, "{:.2f}")
    vs_book(f"{name}: F1(neg)", f1n, b_f1n, "{:.2f}")
    vs_book(f"{name}: dims", X_train[name].shape[1], b_dims, "{:,}")

### round 1: counting
                      accuracy   macro-F1   F1(neg)      dims      fit
majority baseline        93.68      48.37      0.00         —        —


bag of words             92.14      73.51     51.30    30,195     0.7 s


tf-idf                   90.07      71.95     49.41    30,195     0.5 s


tf-idf 1-2 gram          92.57      76.21     56.47   101,659     1.0 s


tf-idf char_wb 3-5       88.63      70.39     47.14    94,939     3.4 s

bag of words: accuracy                 this run      92.14   book      92.14   same as the book
bag of words: macro-F1                 this run      73.51   book      73.51   same as the book
bag of words: F1(neg)                  this run      51.30   book      51.30   same as the book
bag of words: dims                     this run     30,195   book     30,195   same as the book
tf-idf: accuracy                       this run      90.07   book      90.07   same as the book
tf-idf: macro-F1                       this run      71.95   book      71.95   same as the book
tf-idf: F1(neg)                        this run      49.41   book      49.41   same as the book
tf-idf: dims                           this run     30,195   book     30,195   same as the book
tf-idf 1-2 gram: accuracy              this run      92.57   book      92.57   same as the book
tf-idf 1-2 gram: macro-F1              this run      76.21   bo

**Nasıl okunur.** Bu derlemde ve bu ayarlarla en yüksek macro-F1, kelime unigram ve
bigram'ları üzerindeki TF-IDF'in; düz sayımlar da TF-IDF'in üzerinde. Bigram öznitelik
sayısı olan 101.659, [12. bölümün](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/12-siniflandirma.html) bölmeyi birebir geri bulmak için kullandığı parmak
izi. Komşu satırlar arasındaki farklar bir ile üç puan arasında; 7. kısım bunlara aralık
koyuyor.

Kelime çözümleyicisi ile karakter çözümleyicisi aynı metinde farklı şeyler görüyor.
Bölümün örneği:

In [10]:
sample = "5/5 harika!! ürün çok i̇yi 👍"
word_analyzer = schemes["tf-idf"].build_analyzer()
char_analyzer = schemes["tf-idf char_wb 3-5"].build_analyzer()
print("### the same string through the two analyzers")
print(f'input:  "{sample}"\n')
print('analyzer="word", token_pattern=[a-zçğıöşü]+')
print(" ", word_analyzer(sample))
print('\nanalyzer="char_wb", ngram_range=(3, 5)     (first few of the 3-grams)')
three_grams = [g for g in char_analyzer(sample) if len(g) == 3]
print(" ", three_grams[:11], "...")
print()
for label, ours, book in [("word tokens", word_analyzer(sample), ['harika', 'ürün', 'çok', 'i', 'yi']),
                          ("first eleven char 3-grams", three_grams[:11],
                           [' 5/', '5/5', '/5 ', ' ha', 'har', 'ari', 'rik', 'ika', 'ka!', 'a!!', '!! '])]:
    print(f"{label}: " + ("same as the book" if ours == book else "DIFFERS from the book"))

### the same string through the two analyzers
input:  "5/5 harika!! ürün çok i̇yi 👍"

analyzer="word", token_pattern=[a-zçğıöşü]+
  ['harika', 'ürün', 'çok', 'i', 'yi']

analyzer="char_wb", ngram_range=(3, 5)     (first few of the 3-grams)
  [' 5/', '5/5', '/5 ', ' ha', 'har', 'ari', 'rik', 'ika', 'ka!', 'a!!', '!! '] ...

word tokens: same as the book
first eleven char 3-grams: same as the book


### Bu derlemde fark etmedi

Türkçeye duyarlı küçük harf dönüşümü burada neredeyse hiçbir şeyi değiştirmedi ve bölüm
nedenini ölçüyor: yorumlarda büyük İ neredeyse yok, ama *dosyanın kendisi* birleştirici
noktayı zaten içeriyor; metin yayımlanmadan önce yanlış kuralla küçük harfe çevrilmiş.

**Neye bakacaksınız:** kaç yorumun U+0307 içerdiği ve kaç sözlük sütununun yalnızca bu
işaret bir kelimeyi ikiye böldüğü için var olduğu.

In [11]:
DOT = "\u0307"
all_text = "".join(df["text"])
vs_book("characters in the de-duplicated file", len(all_text), 37_076_865, "{:,}")
vs_book("uppercase characters", sum(c.isupper() for c in all_text), 0)
del all_text

has_dot = sub["text"].str.contains(DOT)
vs_book("reviews with U+0307 in the 50,000", int(has_dot.sum()), 4_194, "{:,}")
vs_book("  share, %", 100 * has_dot.mean(), 8.4, "{:.1f}")
after_i = sum(s_[i - 1] == "i" for s_ in sub["text"] for i, c in enumerate(s_) if c == DOT and i > 0)
total_marks = sum(s_.count(DOT) for s_ in sub["text"])
print(f"U+0307 occurrences: {total_marks:,}, of which after the letter i: {after_i:,}")

vocab = set(schemes["bag of words"].vocabulary_)
repaired = CountVectorizer(min_df=2, token_pattern=PAT, **LOC).fit(train["text"].str.replace(DOT, ""))
only_original = vocab - set(repaired.vocabulary_)
only_repaired = set(repaired.vocabulary_) - vocab
print()
vs_book("vocabulary, text as distributed", len(vocab), 30_195, "{:,}")
vs_book("vocabulary, U+0307 deleted", len(repaired.vocabulary_), 29_823, "{:,}")
vs_book("columns that exist only with the mark", len(only_original), 595, "{:,}")
vs_book("forms that appear only after the repair", len(only_repaired), 223, "{:,}")
print("  fragments:", ", ".join(sorted(only_original)[:10]))
print("  restored: ", ", ".join(sorted(only_repaired)[:10]))

example = next(s_ for s_ in train["text"] if "tavsi" + DOT + "ye" in s_)
print(f"\n  {example[:60]!r}\n  -> {tokenize(example)[:10]}")

characters in the de-duplicated file   this run 37,076,865   book 37,076,865   same as the book


uppercase characters                   this run          0   book          0   same as the book
reviews with U+0307 in the 50,000      this run      4,194   book      4,194   same as the book
  share, %                             this run        8.4   book        8.4   same as the book
U+0307 occurrences: 17,736, of which after the letter i: 17,736



vocabulary, text as distributed        this run     30,195   book     30,195   same as the book
vocabulary, U+0307 deleted             this run     29,823   book     29,823   same as the book
columns that exist only with the mark  this run        595   book        595   same as the book
forms that appear only after the repair this run        223   book        223   same as the book
  fragments: ade, adem, ai, alabi, alinabi, alişveri, annemi, ari, arni, arçeli
  restored:  abimi, alişverişe, alişverişim, alişverişin, alişverişler, annemgile, babylis, bahsettikleri, bebeğimizde, bebişimi

  'si̇pari̇ş verdi̇ği̇m gün kargoya veri̇ldi̇. türki̇ye ni̇n bi'
  -> ['si', 'pari', 'ş', 'verdi', 'ği', 'm', 'gün', 'kargoya', 'veri', 'ldi']


**Nasıl okunur.** `tr_lower`, metinde zaten bulunan bir birleştirici noktayı silmiyor.
Onarım, size verilen metinden işareti silmek; küçük harfe farklı biçimde çevirmek değil.
Onarımın skorlara ne yaptığı bölümün 7. alıştırması ve bu notebook'un sonundaki 2.
alıştırma.

### Her seferinde tek değişiklik

"TF-IDF" aynı anda üç karar demek: IDF ağırlığı, terim frekansının logaritması ve her
satırın L2 normalleştirmesi. Birinci rauntta düz sayımlar TF-IDF'ten yüksek skor aldı;
bölüm bu yüzden aynı unigram sözlüğü üzerinde her seferinde tek bir kararı değiştiriyor.

**Neye bakacaksınız:** düşüş IDF ile değil, L2 normalleştirmesiyle geliyor.
Sınıflandırıcının düzenlileştirme gücü `C` her satırda 1,0'da sabit ve normalleştirilmiş
satırların değerleri çok daha küçük.

In [12]:
C_train = X_train["bag of words"].astype(np.float64)
C_test = X_test["bag of words"].astype(np.float64)

def log_tf(X):
    X = X.copy()
    X.data = 1 + np.log(X.data)
    return X

idf = TfidfTransformer(norm=None, use_idf=True, sublinear_tf=False).fit(C_train)
idf_log = TfidfTransformer(norm=None, use_idf=True, sublinear_tf=True).fit(C_train)
ablation = [
    ('raw counts  (= "bag of words")', "no ", "no ", "none", C_train, C_test, (92.14, 73.51, 51.30)),
    ("counts + L2", "no ", "no ", " L2 ", normalize(C_train), normalize(C_test), (88.07, 69.57, 45.85)),
    ("log-TF + L2", "no ", "yes", " L2 ", normalize(log_tf(C_train)), normalize(log_tf(C_test)), (88.37, 70.00, 46.53)),
    ("counts + idf + L2", "yes", "no ", " L2 ", normalize(idf.transform(C_train)), normalize(idf.transform(C_test)), (90.20, 72.16, 49.74)),
    ("log-TF + idf, no norm", "yes", "yes", "none", idf_log.transform(C_train), idf_log.transform(C_test), (92.67, 71.83, 47.61)),
    ('log-TF + idf + L2  (= "tf-idf")', "yes", "yes", " L2 ", normalize(idf_log.transform(C_train)), normalize(idf_log.transform(C_test)), (90.07, 71.95, 49.41)),
]
print("### one change at a time, unigram word features, min_df=2")
print(f"{'':<34}idf   log-TF   norm   accuracy  macro-F1  F1(neg)   book macro-F1")
for name, use_idf, use_log, norm, A, B, book in ablation:
    acc, macro, f1n = scores(y_test, new_clf().fit(A, y_train).predict(B))
    same = "same" if f"{macro:.2f}" == f"{book[1]:.2f}" and f"{acc:.2f}" == f"{book[0]:.2f}" else "DIFFERS"
    print(f"{name:<34}{use_idf}    {use_log}     {norm}  {acc:8.2f}  {macro:8.2f}  {f1n:7.2f}   {book[1]:8.2f}  {same}")

### one change at a time, unigram word features, min_df=2
                                  idf   log-TF   norm   accuracy  macro-F1  F1(neg)   book macro-F1


raw counts  (= "bag of words")    no     no      none     92.14     73.51    51.30      73.51  same
counts + L2                       no     no       L2      88.07     69.57    45.85      69.57  same


log-TF + L2                       no     yes      L2      88.37     70.00    46.53      70.00  same
counts + idf + L2                 yes    no       L2      90.20     72.16    49.74      72.16  same
log-TF + idf, no norm             yes    yes     none     92.67     71.83    47.61      71.83  same
log-TF + idf + L2  (= "tf-idf")   yes    yes      L2      90.07     71.95    49.41      71.95  same


Demek ki adil karşılaştırma her temsile kendi `C` değerini veriyor. Bölüm bu değeri eğitim
kümesinden ayrılan bir doğrulama bölmesinde seçiyor (32.000 satırla eğit, 8.000 satırla
doğrula), sonra seçilen değerle 40.000 satırın tamamında yeniden eğitiyor ve test kümesini
bir kez puanlıyor.

**Neye bakacaksınız:** iki satırın sırası tersine dönüyor. `C` her temsil için ayrı
seçildiğinde TF-IDF düz sayımların üzerine çıkıyor.

In [13]:
C_GRID = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0]
fit_part, val_part = train_test_split(train, test_size=8_000, stratify=train["label"], random_state=42)
tuned_pred = {}
print("### C chosen on a validation split carved out of train (32,000 fit / 8,000 val)")
for name, book_C, book_f1 in [("bag of words", 1.0, 73.51), ("tf-idf", 10.0, 74.71)]:
    vec = make_schemes()[name]
    X_fit, X_val = vec.fit_transform(fit_part["text"]), vec.transform(val_part["text"])
    val_f1 = {C: f1_score(val_part["label"], new_clf(C).fit(X_fit, fit_part["label"]).predict(X_val),
                          average="macro") for C in C_GRID}
    best_C = max(val_f1, key=val_f1.get)
    print(f"{name}: validation macro-F1 by C  " + "  ".join(f"{C:g}:{100 * v:.2f}" for C, v in val_f1.items()))
    tuned_pred[name] = new_clf(best_C).fit(X_train[name], y_train).predict(X_test[name])
    vs_book(f"  {name}: best C on validation", best_C, book_C)
    vs_book(f"  {name}: test macro-F1", scores(y_test, tuned_pred[name])[1], book_f1, "{:.2f}")

### C chosen on a validation split carved out of train (32,000 fit / 8,000 val)


bag of words: validation macro-F1 by C  0.01:62.88  0.03:66.27  0.1:68.98  0.3:70.63  1:71.63  3:70.84  10:70.11  30:69.54  100:69.74


  bag of words: best C on validation   this run        1.0   book        1.0   same as the book
  bag of words: test macro-F1          this run      73.51   book      73.51   same as the book


tf-idf: validation macro-F1 by C  0.01:58.10  0.03:60.75  0.1:64.59  0.3:67.89  1:70.71  3:71.58  10:72.22  30:71.61  100:71.10
  tf-idf: best C on validation         this run       10.0   book       10.0   same as the book
  tf-idf: test macro-F1                this run      74.71   book      74.71   same as the book


Bu 1,2 puanlık fark, test kümesinin örnekleme gürültüsünden büyük mü? İki model aynı
10.000 yorumda puanlandı, dolayısıyla karşılaştırma *eşleştirilmiş*: test yorumlarını
yerine koyarak yeniden örnekleyin, her örneklemde iki modeli de puanlayın ve farkın
dağılımına bakın. Bu notebook'taki her aralık `paired_bootstrap` ile hesaplanıyor. İşlev
her modelin karışıklık matrisindeki dört sayım üzerinden çalışıyor, bu yüzden 10.000
yeniden örnekleme birkaç saniye sürüyor. Yeniden örneklenen konumlar notebook'un geri
kalanı boyunca bellekte kalıyor (yaklaşık 400 MB).

McNemar'ın kesin testi yalnızca iki modelin doğru olup olmama konusunda ayrıştığı yorumlara
bakıyor.

In [14]:
# 10.000 test konumunun N_BOOT yeniden örneklemi, bölümün çektiği biçimde; int32 olarak tutulur
BOOT_IDX = np.random.default_rng(SEED).integers(0, len(y_test), (N_BOOT, len(y_test))).astype(np.int32)


def macro_f1_boot(y_true, y_pred, idx=BOOT_IDX, chunk=500):
    """y_pred'in her bootstrap örneklemindeki macro-F1'i (yüzde), karışıklık sayımlarından."""
    cell = (2 * y_true + y_pred).astype(np.int8)       # 0: ikisi de olumsuz, 1: (olumsuz, olumlu), 2: (olumlu, olumsuz), 3: ikisi de olumlu
    out = np.empty(len(idx))
    for start in range(0, len(idx), chunk):
        c = cell[idx[start:start + chunk]]
        tn, fp, fn, tp = [(c == k).sum(axis=1) for k in range(4)]
        with np.errstate(divide="ignore", invalid="ignore"):
            f1_pos = np.where(2 * tp + fp + fn > 0, 2 * tp / (2 * tp + fp + fn), 0.0)
            f1_neg = np.where(2 * tn + fn + fp > 0, 2 * tn / (2 * tn + fn + fp), 0.0)
        out[start:start + chunk] = 100 * (f1_pos + f1_neg) / 2
    return out


def paired_bootstrap(y_true, pred_a, pred_b):
    """Gözlenen macro-F1 farkı a - b ve %95'lik bootstrap aralığı."""
    d = macro_f1_boot(y_true, pred_a) - macro_f1_boot(y_true, pred_b)
    observed = scores(y_true, pred_a)[1] - scores(y_true, pred_b)[1]
    return observed, np.percentile(d, 2.5), np.percentile(d, 97.5)


a, b = tuned_pred["tf-idf"], tuned_pred["bag of words"]
only_a = int(((a == y_test) & (b != y_test)).sum())
only_b = int(((a != y_test) & (b == y_test)).sum())
observed, low, high = paired_bootstrap(y_test, a, b)
note = "fewer resamples" if SKIP_HEAVY else None
print(f"### tf-idf (C chosen) against raw counts (C chosen), same 10,000 test reviews ({N_BOOT:,} resamples)")
vs_book("only tf-idf right", only_a, 151)
vs_book("only raw counts right", only_b, 80)
print(f"exact McNemar test on the two counts: p = {binomtest(only_a, only_a + only_b, 0.5).pvalue:.1e}   (book: p < 0.001)")
vs_book("macro-F1 difference", observed, 1.20, "{:.2f}")
vs_book("95% interval, lower end", low, 0.42, "{:.2f}", note=note)
vs_book("95% interval, upper end", high, 1.99, "{:.2f}", note=note)

### tf-idf (C chosen) against raw counts (C chosen), same 10,000 test reviews (10,000 resamples)
only tf-idf right                      this run        151   book        151   same as the book
only raw counts right                  this run         80   book         80   same as the book
exact McNemar test on the two counts: p = 3.5e-06   (book: p < 0.001)
macro-F1 difference                    this run       1.20   book       1.20   same as the book
95% interval, lower end                this run       0.42   book       0.42   same as the book
95% interval, upper end                this run       1.99   book       1.99   same as the book


### Model ne öğrendi, kelimelerle

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/04-karsilastirma.html#model-ne-öğrendi-kelimelerle)

Kelime öznitelikleri üzerindeki doğrusal bir model okunabilir. Bölüm, unigram TF-IDF
modelinin (`C = 1.0`, birinci rauntun satırı) en büyük katsayılarını basıyor.

**Neye bakacaksınız:** olumsuz taraf olumsuzlanmış fiillerle dolu (`etmiyorum`, `etmem`,
`beğenmedim`, `kalmadım`). Türkçe olumsuzlamayı kelimenin içine koyuyor; kelime düzeyindeki
bir model de gördüğü her olumsuz biçim için ayrı bir öznitelik ediniyor.

In [15]:
unigram_model = new_clf().fit(X_train["tf-idf"], y_train)
terms = schemes["tf-idf"].get_feature_names_out()
weights = unigram_model.coef_[0]                  # classes_ [0, 1]: pozitif ağırlık -> etiket 1
order = np.argsort(weights)

BOOK_POS = [("gayet", 8.04), ("ederim", 6.94), ("mükemmel", 6.13), ("teşekkürler", 6.09), ("harika", 5.93),
            ("memnunum", 5.84), ("güzel", 5.48), ("süper", 4.98), ("kaliteli", 4.32)]
BOOK_NEG = [("etmiyorum", -8.16), ("değil", -6.95), ("kötü", -6.63), ("etmem", -6.35), ("beğenmedim", -6.21),
            ("iade", -5.85), ("fakat", -5.52), ("kalitesiz", -5.47), ("kalmadım", -5.33)]
print("### tf-idf + logistic regression, largest coefficients")
print(f"{'towards POSITIVE':<24}{'towards NEGATIVE'}")
for i in range(12):
    p, n = order[-1 - i], order[i]
    print(f"{terms[p]:<12} {weights[p]:+6.2f}     {terms[n]:<12} {weights[n]:+6.2f}")
print()
lookup = dict(zip(terms, weights))
same = sum(f"{lookup[w]:+.2f}" == f"{c:+.2f}" for w, c in BOOK_POS + BOOK_NEG)
print(f"the chapter prints 18 of these coefficients; {same} of 18 are the same at two decimals")

### tf-idf + logistic regression, largest coefficients
towards POSITIVE        towards NEGATIVE
gayet         +8.04     etmiyorum     -8.16
ederim        +6.94     değil         -6.95
mükemmel      +6.13     kötü          -6.63
teşekkürler   +6.09     etmem         -6.35
harika        +5.93     beğenmedim    -6.21
memnunum      +5.84     iade          -5.85
gerçekten     +5.79     fakat         -5.52
güzel         +5.48     kalitesiz     -5.47
süper         +4.98     kalmadım      -5.33
ediyorum      +4.86     zayıf         -4.78
rahat         +4.52     almayın       -4.62
kaliteli      +4.32     ama           -4.28

the chapter prints 18 of these coefficients; 18 of 18 are the same at two decimals


## 6. İkinci raunt: 3. bölümün yöntemleri

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/04-karsilastirma.html#raunt-3.-bölümün-yöntemleri)

Üç embedding modeli; her biri bölümün ayarlarıyla, yalnızca 40.000 eğitim yorumu üzerinde
eğitiliyor. Bir yorum 100 sayılık tek bir vektöre dönüşüyor:

- **ortalaması alınmış word2vec**: yorumun sözlükte bulunan kelimelerinin vektörlerinin
  ortalaması; hiç kelimesi bulunmayan yorum sıfır vektörünü alıyor;
- **ortalaması alınmış fastText**: yorumun *bütün* kelimeleri üzerinden ortalama, çünkü
  fastText görmediği bir kelimenin vektörünü karakter n-gram'larından kuruyor;
- **doc2vec (PV-DBOW)**: her eğitim yorumu için öğrenilen, her test yorumu için ise
  çıkarımla bulunan bir vektör.

Ardından vektörleri aynı lojistik regresyon okuyor.

**Yeniden üretilebilirlik.** Bölüm ilk koşusunun tablosunu basıyor ve aynı kodun aynı
tohumla ikinci bir koşusunun 61,37, 61,29 ve 55,44 macro-F1 verdiğini söylüyor. Hücre
ikisiyle de karşılaştırıyor. Bu notebook'un kaydedildiği ortamda word2vec ve fastText
bölümün *ikinci* koşusunu birebir yeniden üretiyor; doc2vec ise notebook her
çalıştırıldığında değişiyor. doc2vec'teki nedenin kaynağı `gensim` 4.4.0'ın kendisi:
`infer_vector` her test belgesini, belgenin kelimelerinin Python'daki yerleşik `hash`
değeriyle tohumlanan bir vektörden başlatıyor; Python da `PYTHONHASHSEED` ayarlanmadıkça
string hash'lerini her süreçte rastgele seçiyor. Yani doc2vec satırı tek bir çekim.
Eğitim yaklaşık bir dakika sürüyor.

`quiet_gensim`, `gensim` 4.4.0'ın eğitim sırasında bazen bastığı bir notu topluyor
(`Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'`). Ortada bir Python
istisnası yok ve eğitim etkilenmiyor; nedenini 3. bölümün notebook'u açıklıyor.

In [16]:
import contextlib
import io

from gensim.models import FastText, Word2Vec
from gensim.models.doc2vec import Doc2Vec, TaggedDocument


@contextlib.contextmanager
def quiet_gensim(what):
    """gensim'in 'Exception ignored in our_dot_float' notlarını toplar, yerine sayısını basar."""
    buffer = io.StringIO()
    with contextlib.redirect_stderr(buffer):
        yield
    lines = buffer.getvalue().splitlines()
    notes = sum("Exception ignored in" in line for line in lines)
    if notes:
        print(f"({what}: gensim printed its 'Exception ignored in our_dot_float' note {notes} times)")
    for line in lines:
        if line.strip() and "Exception ignored in" not in line:
            print(line)


def average(kv, docs, use_oov):
    """Belge başına bir satır: kelime vektörlerinin ortalaması (hiç yoksa sıfır vektörü)."""
    X = np.zeros((len(docs), kv.vector_size), dtype=np.float32)
    for i, doc in enumerate(docs):
        words = doc if use_oov else [w for w in doc if w in kv.key_to_index]
        if words:
            X[i] = np.mean([kv[w] for w in words], axis=0)
    return X


timing = {}
with quiet_gensim("round 2"):
    start = time.perf_counter()
    # bölümün üç modeli
    w2v = Word2Vec(train_tok, vector_size=100, window=5, min_count=2,
                   sg=1, negative=5, epochs=10, workers=1, seed=42)
    timing["averaged word2vec"] = time.perf_counter() - start
    start = time.perf_counter()
    ft  = FastText(train_tok, vector_size=100, window=5, min_count=2,
                   sg=1, negative=5, epochs=10, workers=1, seed=42,
                   min_n=3, max_n=6, bucket=500_000)
    timing["averaged fastText"] = time.perf_counter() - start
    start = time.perf_counter()
    d2v = Doc2Vec([TaggedDocument(d, [i]) for i, d in enumerate(train_tok)],
                  vector_size=100, min_count=2, epochs=20, dm=0,
                  negative=5, workers=1, seed=42)
    timing["doc2vec (PV-DBOW)"] = time.perf_counter() - start

    dense_test_seconds = {}
    start = time.perf_counter()
    X_train["averaged word2vec"] = average(w2v.wv, train_tok, use_oov=False)
    t0 = time.perf_counter(); X_test["averaged word2vec"] = average(w2v.wv, test_tok, use_oov=False)
    dense_test_seconds["averaged word2vec"] = time.perf_counter() - t0
    X_train["averaged fastText"] = average(ft.wv, train_tok, use_oov=True)
    t0 = time.perf_counter(); X_test["averaged fastText"] = average(ft.wv, test_tok, use_oov=True)
    dense_test_seconds["averaged fastText"] = time.perf_counter() - t0
    X_train["doc2vec (PV-DBOW)"] = np.vstack([d2v.dv[i] for i in range(len(train_tok))])
    t0 = time.perf_counter(); X_test["doc2vec (PV-DBOW)"] = np.vstack([d2v.infer_vector(d) for d in test_tok])
    dense_test_seconds["doc2vec (PV-DBOW)"] = time.perf_counter() - t0

DENSE = ["averaged word2vec", "averaged fastText", "doc2vec (PV-DBOW)"]
BOOK_ROUND2 = {"averaged word2vec": (79.92, 61.39, 34.64), "averaged fastText": (79.64, 61.36, 34.79),
               "doc2vec (PV-DBOW)": (71.96, 55.13, 27.66)}
BOOK_RERUN = {"averaged word2vec": 61.37, "averaged fastText": 61.29, "doc2vec (PV-DBOW)": 55.44}
dense_clf = {}
print("### round 2: coordinates")
print(f"{'':<22}accuracy   macro-F1   F1(neg)   dims   embed train")
for name in DENSE:
    dense_clf[name] = new_clf().fit(X_train[name], y_train)
    pred[name] = dense_clf[name].predict(X_test[name])
    acc, macro, f1n = scores(y_test, pred[name])
    print(f"{name:<22}{acc:8.2f} {macro:10.2f} {f1n:9.2f} {X_train[name].shape[1]:>6} {timing[name]:9.1f} s")
print(f"\nword2vec vocabulary {len(w2v.wv):,} words")
for name in DENSE:
    acc, macro, f1n = scores(y_test, pred[name])
    tolerance = 1.0 if name.startswith("doc2vec") else 0.5      # doc2vec çıkarımı hash ile tohumlanıyor
    vs_book_near(f"{name}: accuracy", acc, BOOK_ROUND2[name][0], tolerance)
    vs_book_near(f"{name}: macro-F1", macro, BOOK_ROUND2[name][1], tolerance)
    vs_book_near(f"  macro-F1 against the chapter's re-run", macro, BOOK_RERUN[name], tolerance)

(round 2: gensim printed its 'Exception ignored in our_dot_float' note 15 times)
### round 2: coordinates
                      accuracy   macro-F1   F1(neg)   dims   embed train
averaged word2vec        79.86      61.37     34.65    100      15.2 s
averaged fastText        79.56      61.29     34.70    100      32.6 s
doc2vec (PV-DBOW)        72.42      55.54     28.14    100       9.4 s

word2vec vocabulary 30,738 words
averaged word2vec: accuracy            this run      79.86   book      79.92   -0.06 from the book, inside ±0.5
averaged word2vec: macro-F1            this run      61.37   book      61.39   -0.02 from the book, inside ±0.5
  macro-F1 against the chapter's re-run this run      61.37   book      61.37   +0.00 from the book, inside ±0.5
averaged fastText: accuracy            this run      79.56   book      79.64   -0.08 from the book, inside ±0.5
averaged fastText: macro-F1            this run      61.29   book      61.36   -0.07 from the book, inside ±0.5
  macro-F1 ag

**Nasıl okunur.** Bu derlemde ve bu ayarlarla üç embedding satırının üçü de macro-F1'de
dört sayım satırının altında ve çoğunluk kuralının üstünde. word2vec ile fastText arasında
onda bir puandan az fark var.

Yoğun bir satırın ne kadarı eğitim tohumundan geliyor? Bölüm yalnızca embedding tohumunu
değiştirdi (beş tohum) ve aralığı bildiriyor. Sıradaki hücre bunu tekrarlıyor.

**Ağır hücre**: on eğitim, yaklaşık dört dakika; bu notebook'un süresinin yarısından
fazlası. `ANLP_SKIP_HEAVY=1` altında tek tohum eğitiyor.

In [17]:
SEEDS = [1] if SKIP_HEAVY else [1, 2, 3, 4, 5]
spread = {"averaged word2vec": [], "averaged fastText": []}
start = time.perf_counter()
with quiet_gensim("seed spread"):
    for seed in SEEDS:
        m = Word2Vec(train_tok, vector_size=100, window=5, min_count=2,
                     sg=1, negative=5, epochs=10, workers=1, seed=seed)
        p = new_clf().fit(average(m.wv, train_tok, False), y_train).predict(average(m.wv, test_tok, False))
        spread["averaged word2vec"].append(scores(y_test, p)[1])
        m = FastText(train_tok, vector_size=100, window=5, min_count=2,
                     sg=1, negative=5, epochs=10, workers=1, seed=seed,
                     min_n=3, max_n=6, bucket=500_000)
        p = new_clf().fit(average(m.wv, train_tok, True), y_train).predict(average(m.wv, test_tok, True))
        spread["averaged fastText"].append(scores(y_test, p)[1])
print(f"### macro-F1 over embedding seeds {SEEDS}   ({time.perf_counter() - start:.0f}s)")
BOOK_SPREAD = {"averaged word2vec": (61.57, 62.06), "averaged fastText": (60.86, 61.68)}
for name, values in spread.items():
    print(f"{name:<20} " + "  ".join(f"{v:.2f}" for v in values)
          + f"   range {min(values):.2f}–{max(values):.2f}   book {BOOK_SPREAD[name][0]:.2f}–{BOOK_SPREAD[name][1]:.2f}"
          + ("   [one seed only: ANLP_SKIP_HEAVY=1]" if SKIP_HEAVY else ""))

(seed spread: gensim printed its 'Exception ignored in our_dot_float' note 73 times)
### macro-F1 over embedding seeds [1, 2, 3, 4, 5]   (249s)
averaged word2vec    61.58  61.97  62.06  61.70  61.57   range 61.57–62.06   book 61.57–62.06
averaged fastText    61.51  61.21  60.86  61.68  61.31   range 60.86–61.68   book 60.86–61.68


İki raunt tek tabloda, macro-F1'e göre sıralı; bölümün bastığı biçimde.

In [18]:
ALL = ["bag of words", "tf-idf", "tf-idf 1-2 gram", "tf-idf char_wb 3-5"] + DENSE
print("### both rounds, sorted by macro-F1")
for name in sorted(ALL, key=lambda n: -scores(y_test, pred[n])[1]):
    acc, macro, f1n = scores(y_test, pred[name])
    print(f"{name:<22}{acc:8.2f} {macro:10.2f} {f1n:9.2f} {X_train[name].shape[1]:>9,}")
print(f"{'majority baseline':<22}{majority_acc:8.2f} {majority_macro:10.2f} {0:9.2f}         —")

### both rounds, sorted by macro-F1
tf-idf 1-2 gram          92.57      76.21     56.47   101,659
bag of words             92.14      73.51     51.30    30,195
tf-idf                   90.07      71.95     49.41    30,195
tf-idf char_wb 3-5       88.63      70.39     47.14    94,939
averaged word2vec        79.86      61.37     34.65       100
averaged fastText        79.56      61.29     34.70       100
doc2vec (PV-DBOW)        72.42      55.54     28.14       100
majority baseline        93.68      48.37      0.00         —


## 7. Aralıklar: eşleştirilmiş bootstrap

Yukarıdaki tablo tek bir test kümesi. Bölüm, komşu satır çiftlerinin her birinin macro-F1
farkına, 5. kısımdaki eşleştirilmiş bootstrap ile %95'lik bir aralık koyuyor: iki model
için de aynı yeniden örneklenmiş test yorumları.

**Neye bakacaksınız:** hangi aralıkların sıfırı içerdiği. İki seyrek satır arasındaki
çiftler bölümle aynı yeniden örneklemleri kullanıyor (tohum 42) ve birebir yeniden
üretiliyor. Bölüm, yoğun bir satır içeren çiftleri ikinci koşusundan hesapladı; bunlar bu
yüzden bir tolerans içinde karşılaştırılıyor ve doc2vec'li çift o satırla birlikte oynuyor.

In [19]:
BOOK_PAIRS = [("tf-idf 1-2 gram", "bag of words", 2.69, 1.60, 3.82),
              ("bag of words", "tf-idf", 1.56, 0.53, 2.55),
              ("tf-idf", "tf-idf char_wb 3-5", 1.57, 0.59, 2.54),
              ("tf-idf char_wb 3-5", "averaged word2vec", 9.01, 7.97, 10.01),
              ("averaged word2vec", "averaged fastText", 0.08, -0.60, 0.77),
              ("averaged fastText", "doc2vec (PV-DBOW)", 5.85, 5.02, 6.69),
              ("tf-idf 1-2 gram", "averaged word2vec", 14.83, 13.65, 16.01)]
print(f"### paired bootstrap on the 10,000 test reviews: macro-F1 difference, 95% interval ({N_BOOT:,} resamples)")
for a, b, b_obs, b_low, b_high in BOOK_PAIRS:
    observed, low, high = paired_bootstrap(y_test, pred[a], pred[b])
    ours = f"{observed:+6.2f}   [{low:+6.2f}, {high:+6.2f}]"
    book = f"{b_obs:+6.2f}   [{b_low:+6.2f}, {b_high:+6.2f}]"
    if SKIP_HEAVY:
        verdict = "not comparable: fewer resamples"
    elif a in DENSE or b in DENSE:
        verdict = "dense row: " + ("inside" if max(abs(observed - b_obs), abs(low - b_low), abs(high - b_high)) <= 0.5 else "OUTSIDE") + " ±0.5 of the book"
    else:
        verdict = "same as the book" if ours == book else "DIFFERS from the book"
    print(f"{a:<19} - {b:<19} {ours}   book {book}   {verdict}")

### paired bootstrap on the 10,000 test reviews: macro-F1 difference, 95% interval (10,000 resamples)


tf-idf 1-2 gram     - bag of words         +2.69   [ +1.60,  +3.82]   book  +2.69   [ +1.60,  +3.82]   same as the book


bag of words        - tf-idf               +1.56   [ +0.53,  +2.55]   book  +1.56   [ +0.53,  +2.55]   same as the book


tf-idf              - tf-idf char_wb 3-5   +1.57   [ +0.59,  +2.54]   book  +1.57   [ +0.59,  +2.54]   same as the book


tf-idf char_wb 3-5  - averaged word2vec    +9.01   [ +7.97, +10.01]   book  +9.01   [ +7.97, +10.01]   dense row: inside ±0.5 of the book


averaged word2vec   - averaged fastText    +0.08   [ -0.60,  +0.77]   book  +0.08   [ -0.60,  +0.77]   dense row: inside ±0.5 of the book


averaged fastText   - doc2vec (PV-DBOW)    +5.75   [ +4.92,  +6.59]   book  +5.85   [ +5.02,  +6.69]   dense row: inside ±0.5 of the book


tf-idf 1-2 gram     - averaged word2vec   +14.83   [+13.65, +16.01]   book +14.83   [+13.65, +16.01]   dense row: inside ±0.5 of the book


**Nasıl okunur.** İki aile arasındaki fark ve seyrek ailenin içindeki komşu farklar bu test
kümesinin örnekleme gürültüsünden büyük. word2vec ile fastText arasındaki fark değil:
aralığı sıfırı içeriyor. Bu aralıklar yalnızca *test* yorumlarının örneklemesini kapsıyor.
Başka bir eğitim bölmesi hakkında bir şey söylemiyorlar; yoğun satırlarda yukarıda ölçülen
tohum yayılımı da bunun üzerine ekleniyor.

## 8. Sözlük dışı (out-of-vocabulary) sınavı

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/04-karsilastirma.html#sözlük-dışı-out-of-vocabulary-sınavı)

word2vec'in yalnızca tuttuğu biçimler için vektörü var; burada bunlar eğitimde en az iki
kez görülenler. Bölüm, test kümesinin ne kadarının bu sözlüğün dışında kaldığını token'a
ve farklı biçime göre sayıyor. Bu sayımlar yalnızca sözlüğe bağlı, dolayısıyla birebir
yeniden üretiliyor.

**Neye bakacaksınız:** token'ların %7'si ve farklı biçimlerin %46'sı.

In [20]:
kv, fv = w2v.wv, ft.wv
test_tokens = [w for doc in test_tok for w in doc]
test_forms = set(test_tokens)
oov_tokens = sum(w not in kv.key_to_index for w in test_tokens)
oov_forms = sorted(w for w in test_forms if w not in kv.key_to_index)
empty = [i for i, doc in enumerate(test_tok) if not any(w in kv.key_to_index for w in doc)]

print("### coverage of the 10,000 test reviews")
vs_book("test tokens", len(test_tokens), 229_055, "{:,}")
vs_book("outside the word2vec vocabulary", oov_tokens, 16_660, "{:,}")
vs_book("  share of tokens, %", 100 * oov_tokens / len(test_tokens), 7.27, "{:.2f}")
vs_book("distinct test word forms", len(test_forms), 32_573, "{:,}")
vs_book("outside the vocabulary", len(oov_forms), 15_098, "{:,}")
vs_book("  share of forms, %", 100 * len(oov_forms) / len(test_forms), 46.4, "{:.1f}")
vs_book("test reviews with no usable token", len(empty), 7)

never_tokens = sum(counts[w] == 0 for w in test_tokens)
never_forms = sum(counts[w] == 0 for w in test_forms)
print("\n### the same counts, split by whether the form was ever seen in training")
vs_book("test tokens never seen in training", never_tokens, 11_681, "{:,}")
vs_book("  share of tokens, %", 100 * never_tokens / len(test_tokens), 5.10, "{:.2f}")
vs_book("distinct forms never seen", never_forms, 10_921, "{:,}")
vs_book("  share of forms, %", 100 * never_forms / len(test_forms), 33.5, "{:.1f}")
vs_book("seen once, cut by min_count=2", len(oov_forms) - never_forms, 4_177, "{:,}")

### coverage of the 10,000 test reviews
test tokens                            this run    229,055   book    229,055   same as the book
outside the word2vec vocabulary        this run     16,660   book     16,660   same as the book
  share of tokens, %                   this run       7.27   book       7.27   same as the book
distinct test word forms               this run     32,573   book     32,573   same as the book
outside the vocabulary                 this run     15,098   book     15,098   same as the book
  share of forms, %                    this run       46.4   book       46.4   same as the book
test reviews with no usable token      this run          7   book          7   same as the book

### the same counts, split by whether the form was ever seen in training
test tokens never seen in training     this run     11,681   book     11,681   same as the book
  share of tokens, %                   this run       5.10   book       5.10   same as the book
distinct forms never s

Kullanılabilir hiçbir token'ı olmayan yedi yorum, word2vec ortalamasında sıfır vektörüne
dönüşüyor. Bölüm yedisinin de olumlu etiketli olduğunu ve yedisinin de olumsuz
sınıflandırıldığını bildiriyor, çünkü sınıflandırıcının sabit terimi negatif.

In [21]:
zero_vector_p = dense_clf["averaged word2vec"].predict_proba(np.zeros((1, 100)))[0, 1]
print("texts:    ", [test["text"].iloc[i][:30] for i in empty])
print("gold:     ", y_test[empty].tolist())
print("predicted:", pred["averaged word2vec"][empty].tolist())
vs_book_near("intercept", float(dense_clf["averaged word2vec"].intercept_[0]), -0.83, 0.15)
print(f"P(positive | zero vector) = {zero_vector_p:.2f}")

texts:     ['yanilmişim...', 'avantajlari', 'kuşadası', 'murat', 'mukemmelllll', 'selametle ..', 'haaarika!']
gold:      [1, 1, 1, 1, 1, 1, 1]
predicted: [0, 0, 0, 0, 0, 0, 0]
intercept                              this run      -0.83   book      -0.83   +0.00 from the book, inside ±0.15
P(positive | zero vector) = 0.30


fastText herhangi bir biçimin vektörünü karakter n-gram'larından kuruyor, dolayısıyla
sözlükte bulma kapsaması %100. Bölüm, görülmemiş bir biçimin n-gram malzemesinin ne
kadarını modelin gerçekten gördüğünü ölçüyor: sözlük dışı 15.098 biçimin her biri için,
3 ile 6 karakter arasındaki n-gram'larından en az bir sözlük kelimesinde de geçenlerin
payı.

In [22]:
def char_ngrams(word, low=3, high=6):
    s = f"<{word}>"
    return [s[i:i + n] for n in range(low, high + 1) for i in range(len(s) - n + 1)]


seen_ngrams = {g for w in fv.index_to_key for g in char_ngrams(w)}
share_seen = np.array([np.mean([g in seen_ngrams for g in char_ngrams(w)]) for w in oov_forms])
vs_book("mean share of n-grams seen, %", 100 * share_seen.mean(), 84.5, "{:.1f}")
vs_book("forms with every n-gram seen, %", 100 * (share_seen == 1).mean(), 24.7, "{:.1f}")
vs_book("forms with fewer than half seen, %", 100 * (share_seen < 0.5).mean(), 4.8, "{:.1f}")

mean share of n-grams seen, %          this run       84.5   book       84.5   same as the book
forms with every n-gram seen, %        this run       24.7   book       24.7   same as the book
forms with fewer than half seen, %     this run        4.8   book        4.8   same as the book


Şimdi komşular. Önce eğitimde hiç geçmeyen üç biçim, sonra sık geçen iki kelime. Listeler
kosinüse göre en yakın beş ya da altı komşu. Bölümün listeleri ilk koşusundan geliyor ve
o koşunun vektörleri yukarıda eğitilenlerden biraz farklı; bu yüzden hücre bir eşleşme
değil, bölümün komşularından kaçının bu koşunun listesinde bulunduğunu basıyor.

**Neye bakacaksınız:** word2vec görülmemiş bir biçimde `KeyError` veriyor. Sık bir kelime
için word2vec başka gövdelerden ilişkili kelimeler, fastText ise yazımı yakın biçimler
döndürüyor.

In [23]:
def neighbours(model, word, k):
    return [w for w, _ in model.most_similar(word, topn=k)]


def vs_book_list(label, ours, book):
    shared = len(set(ours) & set(book))
    print(f"  {label:<9} {', '.join(ours)}\n  {'':<9} [{shared} of the book's {len(book)} are in this list]")


print("### nearest neighbours for unseen word forms")
BOOK_UNSEEN = {"kargolanmamış": ["kargolanmış", "kargolandı", "kargolanma", "kargolanıp", "kargolu"],
               "tavsiyelerim": ["tavsiyeleri", "tavsiyeler", "tavsiyeederim", "tavsiyedir"],
               "hızlıcana": ["hızlıca", "hızlıda", "hızlısı", "hızlıydı", "hızlı"]}
for word, book in BOOK_UNSEEN.items():
    print(f"{word}   (occurrences in training: {counts[word]})")
    try:
        kv[word]
    except KeyError:
        print("  word2vec  KeyError")
    vs_book_list("fastText", neighbours(fv, word, 5), book)

print("\n### nearest neighbours for a common word")
BOOK_COMMON = {"kargo": (["teslimat", "gönderim", "gönderi", "kargoda", "kargoyla", "kargolama"],
                         ["kargocu", "kargolu", "kargosu", "kargonuz", "kargom", "kargomun"]),
               "kalitesiz": (["dandik", "kullanışsız", "dayanıksız", "uyduruk", "kırılacak"],
                             ["kalitesizmiş", "kalitesizliği", "kalitel", "kalitesi", "kalitesindeki"])}
for word, (book_w2v, book_ft) in BOOK_COMMON.items():
    print(f"{word}   (occurrences in training: {counts[word]:,})")
    vs_book_list("word2vec", neighbours(kv, word, len(book_w2v)), book_w2v)
    vs_book_list("fastText", neighbours(fv, word, len(book_ft)), book_ft)

print(f"\n### kalitesizmiş — in the word2vec vocabulary, seen {counts['kalitesizmiş']} times in training")
print("  word2vec ", ", ".join(neighbours(kv, "kalitesizmiş", 5)))
print("  fastText ", ", ".join(neighbours(fv, "kalitesizmiş", 4)))

### nearest neighbours for unseen word forms
kargolanmamış   (occurrences in training: 0)
  word2vec  KeyError
  fastText  kargolanmış, kargolandı, kargolanma, kargolanıp, kargolu
            [5 of the book's 5 are in this list]
tavsiyelerim   (occurrences in training: 0)
  word2vec  KeyError
  fastText  tavsiyeleri, tavsiyeler, tavsiyeederim, şüpelerim, yeğlerim
            [3 of the book's 4 are in this list]
hızlıcana   (occurrences in training: 0)
  word2vec  KeyError
  fastText  hızlıca, hızlıda, hızlısı, hızlıydı, hızlı
            [5 of the book's 5 are in this list]

### nearest neighbours for a common word
kargo   (occurrences in training: 2,590)
  word2vec  teslimat, gönderim, gönderi, kargoda, kargoyla, kargolama
            [6 of the book's 6 are in this list]
  fastText  kargocu, kargolu, kargosu, kargonuz, kargom, kargomun
            [6 of the book's 6 are in this list]
kalitesiz   (occurrences in training: 152)
  word2vec  dandik, kullanışsız, dayanıksız, dikişler, uydu

**Nasıl okunur.** `kalitesizmiş` word2vec sözlüğünün içinde, dolayısıyla sorgu hata
vermiyor; ama biçim iki kez görüldü ve word2vec komşuları ne kelimenin duygusunu ne de
biçimini yansıtıyor. Bölümün vurguladığı şu: sözlük sınırının ötesinde hata gürültülü,
sınırın hemen içinde ise sessiz.

İki kelime bir örnek. Bölüm yazım eğilimini, en az altı harfli ve eğitimde en az 20 kez
geçen sözlük kelimelerinden çekilen 500 kelime üzerinde ölçüyor: her modelin ilk 5
komşusundan, sorgunun ilk beş (ya da dört) karakterini paylaşanların payı.

In [24]:
eligible = [w for w in kv.index_to_key if len(w) >= 6 and counts[w] >= 20]
sample_words = list(np.random.default_rng(42).choice(eligible, 500, replace=False))
vs_book("eligible vocabulary words", len(eligible), 2_806, "{:,}")
BOOK_PREFIX = {(5, "word2vec"): 6.5, (5, "fastText"): 60.7, (4, "word2vec"): 8.6, (4, "fastText"): 67.4}
for k in (5, 4):
    for name, model in [("word2vec", kv), ("fastText", fv)]:
        share = np.mean([np.mean([n[:k] == w[:k] for n in neighbours(model, w, 5)]) for w in sample_words])
        vs_book_near(f"first {k} characters shared, {name}, %", 100 * share, BOOK_PREFIX[(k, name)], 3.0, "{:.1f}")

eligible vocabulary words              this run      2,806   book      2,806   same as the book


first 5 characters shared, word2vec, % this run        6.5   book        6.5   -0.02 from the book, inside ±3.0


first 5 characters shared, fastText, % this run       60.7   book       60.7   -0.02 from the book, inside ±3.0


first 4 characters shared, word2vec, % this run        8.6   book        8.6   +0.04 from the book, inside ±3.0


first 4 characters shared, fastText, % this run       67.4   book       67.4   +0.04 from the book, inside ±3.0


## 9. Skor farkını anlamak için hata örnekleri

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/04-karsilastirma.html#yoğun-yöntemler-neden-kaybetti)

Tablo, sayım yöntemlerinin burada daha yüksek skor aldığını söylüyor; nedenini söylemiyor.
Bölüm, düz kelime TF-IDF'i ile ortalaması alınmış fastText'in *ayrıştığı* test yorumlarına
bakıyor. İki sınıfta, ikisi ayrıştığında tam olarak biri haklı; McNemar testinin kullandığı
da bu iki sayım.

**Neye bakacaksınız:** ayrışmaların nasıl bölündüğü, sonra da her gruptaki yorumların kim
olduğu.

In [25]:
a, b = pred["tf-idf"], pred["averaged fastText"]
only_tfidf = (a == y_test) & (b != y_test)
only_ft = (b == y_test) & (a != y_test)
n_a, n_b = int(only_tfidf.sum()), int(only_ft.sum())
chi2 = (abs(n_a - n_b) - 1) ** 2 / (n_a + n_b)
observed, low, high = paired_bootstrap(y_test, a, b)

vs_book_near("same label, % of test reviews", 100 * (a == b).mean(), 84.1, 0.5, "{:.1f}")
vs_book_near("only tf-idf right", n_a, 1_315, 30, "{:,}")
vs_book_near("only fastText right", n_b, 272, 30, "{:,}")
vs_book_near("McNemar chi-squared (corrected)", chi2, 684.2, 40, "{:.1f}")
vs_book_near("macro-F1 gap", observed, 10.66, 0.5)
print(f"  paired bootstrap 95% [{low:.2f}, {high:.2f}]   book [9.52, 11.77]")

tfidf_vocab = schemes["tf-idf"].vocabulary_
n_tok = np.array([len(doc) for doc in test_tok])
n_oov = np.array([sum(w not in tfidf_vocab for w in doc) for doc in test_tok])
print(f"\n{'':<22}{'reviews':>8}{'gold positive':>16}{'tokens outside the tf-idf vocabulary':>40}")
for name, mask in [("all test reviews", np.ones(len(y_test), bool)), ("both right", (a == y_test) & (b == y_test)),
                   ("only tf-idf right", only_tfidf), ("only fastText right", only_ft)]:
    print(f"{name:<22}{mask.sum():>8,}{100 * y_test[mask].mean():>15.1f}%{100 * n_oov[mask].sum() / n_tok[mask].sum():>39.2f}%")

same label, % of test reviews          this run       84.0   book       84.1   -0.05 from the book, inside ±0.5
only tf-idf right                      this run      1,323   book      1,315   +8.00 from the book, inside ±30
only fastText right                    this run        272   book        272   +0.00 from the book, inside ±30
McNemar chi-squared (corrected)        this run      691.2   book      684.2   +7.02 from the book, inside ±40
macro-F1 gap                           this run      10.66   book      10.66   +0.00 from the book, inside ±0.5
  paired bootstrap 95% [9.52, 11.77]   book [9.52, 11.77]

                       reviews   gold positive    tokens outside the tf-idf vocabulary
all test reviews        10,000           93.7%                                   7.32%
both right               7,684           94.0%                                   6.51%
only tf-idf right        1,323           98.0%                                   9.48%
only fastText right        272      

**Nasıl okunur.** Sözlük dışı biçimler, ayrışılan yorumlarda iki modelin de doğru bildiği
yorumlardakinden daha sık; ama iki tarafta da yaklaşık aynı sıklıkta, yani hangi modelin
haklı çıktığını ayırmıyorlar. Sınıf ise ayırıyor: yalnızca fastText'in doğru bildiği
yorumlar, test kümesinin bütününe göre çok daha sık olumsuz; yalnızca TF-IDF'in doğru
bildiği yorumların ise neredeyse hepsi olumlu. Bölüm bunu, ortalaması alınmış embedding
düzeninin olumsuz sınıfta daha yüksek recall ve daha düşük precision ile çalışması olarak
okuyor.

Bölüm seçilmiş altı ayrışmayı basıyor. Aşağıda aynı altı metin ve iki modelin *bu*
koşudaki tahminleri var (0 = olumsuz, 1 = olumlu). Bölüm bunları ilk koşusundan seçti;
burada iki sütunun aynı olduğu bir satır orada bir ayrışmaydı. Seçilmiş altı örnek metin
türlerini gösterir, oranları değil.

In [26]:
BOOK_EXAMPLES = ["en guvenli alısveriş sitesi", "çok hizli ve temi̇z tesli̇mat", "çok sessiz çalışıyor makina .",
                 "sıkıcı edebi yonu cok zayif", "koku ne çok kötü ne çok iyi", "cok guzel! kedi bayildi buna"]
texts = test["text"].tolist()
print(f"{'text':<34}{'gold':>5}{'tf-idf':>8}{'fastText':>10}")
for example in BOOK_EXAMPLES:
    if example in texts:
        i = texts.index(example)
        print(f"{example:<34}{y_test[i]:>5}{a[i]:>8}{b[i]:>10}")
    else:
        print(f"{example:<34}  not found as a whole test review")

text                               gold  tf-idf  fastText
en guvenli alısveriş sitesi           1       0         1
çok hizli ve temi̇z tesli̇mat         1       0         1
çok sessiz çalışıyor makina .         1       0         1
sıkıcı edebi yonu cok zayif           0       0         1
koku ne çok kötü ne çok iyi           0       0         1
cok guzel! kedi bayildi buna          1       1         1


### Bir yorum ve olumsuzlanmış hâli

Bölüm temsilleri doğrudan sınıyor: beş yorum ve olumsuzlanmış hâlleri, ve her çiftin iki
üyesi arasındaki kosinüs.

**Neye bakacaksınız:** ortalaması alınmış fastText'te bir yorum ile olumsuzlanmış hâli
birbirine yakın. TF-IDF'te çok daha uzaklar, çünkü olumsuzlanmış fiil ayrı bir sütun.
TF-IDF sütunu birebir yeniden üretiliyor; fastText sütunu eğitilen vektörlere bağlı.

In [27]:
def doc_vector(text, model):
    return np.mean([model[w] for w in tokenize(text)], axis=0)


def cos(x, y):
    return float(x @ y / (np.linalg.norm(x) * np.linalg.norm(y)))


PAIRS = [("ürünü çok beğendim", "ürünü hiç beğenmedim", 0.776, 0.193),
         ("kesinlikle tavsiye ederim", "kesinlikle tavsiye etmem", 0.919, 0.474),
         ("kargo çok hızlı geldi", "kargo çok yavaş geldi", 0.943, 0.572),
         ("fiyatına göre gayet kaliteli", "fiyatına göre hiç kaliteli değil", 0.892, 0.690),
         ("bu üründen memnun kaldım", "bu üründen memnun kalmadım", 0.962, 0.582)]
print("### cosine between a review and its negation")
print(f"{'':<62}{'fastText':>9}{'book':>7}{'tf-idf':>9}{'book':>7}")
for positive, negative, book_ft, book_tfidf in PAIRS:
    T = schemes["tf-idf"].transform([positive, negative])
    c_ft = cos(doc_vector(positive, fv), doc_vector(negative, fv))
    c_tfidf = cosine_similarity(T[0], T[1])[0, 0]
    flag = "" if f"{c_tfidf:.3f}" == f"{book_tfidf:.3f}" else "  tf-idf DIFFERS"
    print(f"{positive + ' / ' + negative:<62}{c_ft:>9.3f}{book_ft:>7.3f}{c_tfidf:>9.3f}{book_tfidf:>7.3f}{flag}")

### cosine between a review and its negation
                                                               fastText   book   tf-idf   book
ürünü çok beğendim / ürünü hiç beğenmedim                         0.776  0.776    0.193  0.193
kesinlikle tavsiye ederim / kesinlikle tavsiye etmem              0.919  0.919    0.474  0.474
kargo çok hızlı geldi / kargo çok yavaş geldi                     0.943  0.943    0.572  0.572
fiyatına göre gayet kaliteli / fiyatına göre hiç kaliteli değil    0.892  0.892    0.690  0.690
bu üründen memnun kaldım / bu üründen memnun kalmadım             0.962  0.962    0.582  0.582


## 10. Kesişim noktası: sıra etiket sayısına nasıl bağlı

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/04-karsilastirma.html#kesişim-noktası-embeddingler-ne-zaman-kazanır)

Buraya kadar her şey 40.000 etiket kullandı. Bölüm karşılaştırmayı aynı test kümesinde,
100 ile 40.000 arasında etiketli yorumla tekrarlıyor.

**Burada protokol önemli.** Temsiller sabit kalıyor: sözlük, IDF ağırlıkları ve
embedding'ler 40.000 eğitim yorumunun tamamının *metninden* öğreniliyor; bunun için etiket
gerekmiyor. Yalnızca sınıflandırıcı daha az etiket görüyor. Bu, elinde çok etiketsiz metin
ve az etiket olan birinin durumu. Temsil de yalnızca etiketli altkümeden öğrenilseydi eğri
başka bir eğri olurdu; bölümün 12. alıştırması bunu istiyor ve
[5. bölümün bütçe ızgarası](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/05-veri-etiketleme.html) o diğer protokolü kullanıyor.

Önce, bölümün ilk tablosundaki gibi, her boyut için tek bir tabakalı etiket örneklemi
(tohum 42).

**Neye bakacaksınız:** 1.000 etiketin altında TF-IDF satırları çoğunluk kuralının skorunda
(48,37) duruyor, embedding satırları durmuyor. 2.500 etikette sıra değişmiş.

In [28]:
CURVE = ["tf-idf", "tf-idf char_wb 3-5", "averaged word2vec", "averaged fastText"]
positions = np.arange(len(y_train))
SPARSE_CSR = {name: X_train[name].tocsr() for name in ["tf-idf", "tf-idf 1-2 gram", "tf-idf char_wb 3-5"]}


def label_sample(n, seed):
    if n == len(y_train):
        return positions
    chosen, _ = train_test_split(positions, train_size=n, stratify=y_train, random_state=seed)
    return chosen


def curve_point(name, chosen):
    X = SPARSE_CSR[name] if name in SPARSE_CSR else X_train[name]
    p = new_clf().fit(X[chosen], y_train[chosen]).predict(X_test[name])
    return scores(y_test, p)[1]


BOOK_CURVE = {100: (48.37, 48.37, 53.51, 53.87), 250: (48.53, 48.84, 61.59, 63.10), 500: (49.28, 50.60, 61.02, 61.16),
              1_000: (55.40, 58.55, 59.52, 59.93), 2_500: (66.61, 68.36, 60.91, 61.19), 5_000: (71.16, 70.47, 61.52, 60.80),
              10_000: (72.55, 71.29, 61.80, 61.56), 20_000: (72.42, 71.26, 60.94, 60.96), 40_000: (71.95, 70.39, 61.39, 61.36)}
print("### macro-F1 by labelled training size (same test set of 10,000)")
print(f"{'n':>7} {'negatives':>10}   tf-idf   tf-idf char   word2vec   fastText    sparse columns   dense, max |Δ| from book")
for n, book in BOOK_CURVE.items():
    chosen = label_sample(n, 42)
    row = [curve_point(name, chosen) for name in CURVE]
    sparse_same = all(f"{row[i]:.2f}" == f"{book[i]:.2f}" for i in (0, 1))
    dense_gap = max(abs(row[i] - book[i]) for i in (2, 3))
    print(f"{n:>7,} {int((y_train[chosen] == 0).sum()):>10,} {row[0]:>8.2f} {row[1]:>13.2f} {row[2]:>10.2f} {row[3]:>10.2f}"
          f"    {'same as the book' if sparse_same else 'DIFFER':<16} {dense_gap:.2f}")

### macro-F1 by labelled training size (same test set of 10,000)
      n  negatives   tf-idf   tf-idf char   word2vec   fastText    sparse columns   dense, max |Δ| from book
    100          6    48.37         48.37      58.23      57.49    same as the book 4.72
    250         16    48.53         48.84      65.21      64.94    same as the book 3.62
    500         32    49.28         50.60      63.21      62.80    same as the book 2.19


  1,000         63    55.40         58.55      61.10      61.10    same as the book 1.58


  2,500        158    66.61         68.36      61.70      61.32    same as the book 0.79
  5,000        316    71.16         70.47      61.52      60.97    same as the book 0.17


 10,000        632    72.55         71.29      61.95      61.69    same as the book 0.15


 20,000      1,264    72.42         71.26      60.94      60.88    same as the book 0.08


 40,000      2,529    71.95         70.39      61.37      61.29    same as the book 0.07


**Nasıl okunur.** İki seyrek sütun birebir yeniden üretiliyor. Yoğun sütunlar bölümün ilk
koşusundan gelen tablosundan farklı, en çok da en küçük boyutlarda: etiketli örneklemde 6
ya da 16 olumsuz yorum varken vektörlerdeki küçük bir fark sınıflandırıcıyı çok oynatıyor.

Boyut başına tek etiket örneklemi tek bir çekim. Bölüm ardından her boyutta **on** tabakalı
etiket örneklemi çekiyor; ortalamayı, en küçük ve en büyük değeri bildiriyor ve kelime
bigram TF-IDF'ini ekliyor.

Bu 400 sınıflandırıcı eğitimi ve yaklaşık on saniye sürüyor, çünkü matrisler bir kez
kuruluyor ve yalnızca satırlar değişiyor. Bölüm bu tabloyu ikinci koşusunun vektörleriyle
hesapladı, dolayısıyla dört sütun da karşılaştırılabilir.

In [29]:
TEN = ["tf-idf", "tf-idf 1-2 gram", "averaged word2vec", "averaged fastText"]
SIZES = [100, 250, 500, 1_000, 1_250, 1_500, 1_750, 2_000, 2_500, 5_000]
N_SAMPLES = 10
BOOK_TEN = {100: (48.49, 48.45, 59.68, 58.66, 0), 250: (48.98, 48.70, 61.69, 60.96, 0), 500: (50.65, 49.17, 61.85, 61.96, 0),
            1_000: (56.57, 51.68, 62.15, 61.96, 0), 1_250: (59.59, 53.43, 62.01, 62.00, 2), 1_500: (61.87, 55.63, 62.06, 61.90, 4),
            1_750: (63.95, 57.81, 61.89, 61.64, 10), 2_000: (65.54, 59.69, 61.70, 61.44, 10),
            2_500: (67.52, 63.42, 61.68, 61.55, 10), 5_000: (70.97, 71.49, 61.48, 61.18, 10)}
start = time.perf_counter()
ten = {(name, n, seed): curve_point(name, label_sample(n, seed))
       for n in SIZES for seed in range(N_SAMPLES) for name in TEN}
print(f"### {N_SAMPLES} label samples per size: mean macro-F1 [min, max]; representations fixed   ({time.perf_counter() - start:.0f}s)")
print(f"{'n':>6}   {'tf-idf':<22}{'tf-idf 1-2 gram':<22}{'word2vec':<22}{'fastText':<22}tf-idf > fastText   book")
for n in SIZES:
    cells = []
    for name in TEN:
        v = [ten[(name, n, s)] for s in range(N_SAMPLES)]
        cells.append(f"{np.mean(v):5.2f} [{min(v):5.2f}, {max(v):5.2f}]")
    wins = sum(ten[("tf-idf", n, s)] > ten[("averaged fastText", n, s)] for s in range(N_SAMPLES))
    print(f"{n:>6,}   " + "".join(f"{c:<22}" for c in cells) + f"{wins:>5} / {N_SAMPLES}         {BOOK_TEN[n][4]:>2} / 10")
same = sum(f"{np.mean([ten[(name, n, s)] for s in range(10)]):.2f}" == f"{BOOK_TEN[n][i]:.2f}"
           for n in SIZES for i, name in enumerate(TEN))
print(f"\nmeans equal to the book's at two decimals: {same} of {len(SIZES) * len(TEN)}")
for n in SIZES:
    for i, name in enumerate(TEN):
        ours = np.mean([ten[(name, n, s)] for s in range(10)])
        if f"{ours:.2f}" != f"{BOOK_TEN[n][i]:.2f}":
            vs_book_near(f"  {n:>5,} labels, {name}", ours, BOOK_TEN[n][i], 0.5)

### 10 label samples per size: mean macro-F1 [min, max]; representations fixed   (7s)
     n   tf-idf                tf-idf 1-2 gram       word2vec              fastText              tf-idf > fastText   book
   100   48.49 [48.35, 48.85]  48.45 [48.37, 48.69]  59.68 [54.63, 61.68]  58.66 [53.80, 61.44]      0 / 10          0 / 10
   250   48.98 [48.35, 49.63]  48.70 [48.37, 49.17]  61.69 [58.58, 65.07]  60.96 [56.55, 64.83]      0 / 10          0 / 10
   500   50.65 [49.46, 51.94]  49.17 [48.51, 49.95]  61.85 [59.63, 64.18]  61.96 [60.47, 63.84]      0 / 10          0 / 10
 1,000   56.57 [55.14, 59.80]  51.68 [50.39, 53.36]  62.15 [61.61, 63.32]  61.96 [61.01, 63.15]      0 / 10          0 / 10
 1,250   59.59 [57.39, 62.41]  53.43 [51.60, 56.44]  62.01 [60.81, 62.91]  62.00 [60.83, 63.29]      2 / 10          2 / 10
 1,500   61.87 [60.50, 65.01]  55.63 [53.54, 58.41]  62.06 [60.69, 63.43]  61.90 [60.36, 63.52]      4 / 10          4 / 10
 1,750   63.95 [62.34, 65.35]  57.81 [56.37, 60.

**Nasıl okunur.** Bölümün koşusunda kelime TF-IDF'i 1.000 etikete kadar on örneklemin
onunda da ortalaması alınmış fastText'in altındaydı, 1.750'den itibaren onunda da
üstündeydi, arada ise bölünmüştü. 40.000 etikette en yüksek satır olan kelime bigram
TF-IDF'inin oraya varması için unigram TF-IDF'ten daha çok etiket gerekiyor: kestirilecek
üç kat sütunu var. Tek bir etiket örneklemine dayanan her sıralamayı `[min, max]`
yayılımıyla birlikte okuyun.

Kesişim noktası bu protokole ait: 40.000 etiketsiz metinden öğrenilen temsiller, sıfırdan
eğitilmiş 100 boyutlu bir embedding ve bu derlem.

## 11. Bunun bedeli ne

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/04-karsilastirma.html#bunun-bedeli-ne)

Bölümün maliyet tablosunda üç sütun var: temsili 40.000 eğitim yorumu üzerinde kurma
süresi, 10.000 test yorumunu vektörleştirme süresi ve eğitim matrisinin belleği. Aşağıdaki
süreler bu makinenin, yukarıdaki tekil koşulardan; bölüm, tekrarlanan koşuların yaklaşık
yüzde on oynadığı konusunda uyarıyor. Bellek sütunu makineye bağlı değil.

In [30]:
def matrix_mb(X):
    if sp.issparse(X):
        return (X.data.nbytes + X.indices.nbytes + X.indptr.nbytes) / 1e6
    return X.nbytes / 1e6


BOOK_MB = {"tf-idf": 9.7, "tf-idf 1-2 gram": 15.4, "tf-idf char_wb 3-5": 144.8,
           "averaged word2vec": 16.0, "averaged fastText": 16.0, "doc2vec (PV-DBOW)": 16.0}
print("### cost, on one CPU core, 40,000 training documents (single run, not repeated)")
print(f"{'':<22}{'build the':>15}{'vectorise':>14}{'training':>12}")
print(f"{'':<22}{'representation':>15}{'10,000 test':>14}{'matrix':>12}{'book':>12}")
for name in BOOK_MB:
    if name in cost:
        build, vectorise = cost[name][0], cost[name][1]
    else:
        build, vectorise = timing[name], dense_test_seconds[name]
    mb = matrix_mb(X_train[name])
    same = "same" if f"{mb:.1f}" == f"{BOOK_MB[name]:.1f}" else "DIFFERS"
    print(f"{name:<22}{build:>13.1f} s{vectorise:>12.2f} s{mb:>9.1f} MB{BOOK_MB[name]:>9.1f} MB  {same}")

### cost, on one CPU core, 40,000 training documents (single run, not repeated)
                            build the     vectorise    training
                       representation   10,000 test      matrix        book
tf-idf                          0.5 s        0.11 s      9.7 MB      9.7 MB  same
tf-idf 1-2 gram                 0.9 s        0.18 s     15.4 MB     15.4 MB  same
tf-idf char_wb 3-5              3.0 s        0.72 s    144.8 MB    144.8 MB  same
averaged word2vec              15.2 s        0.10 s     16.0 MB     16.0 MB  same
averaged fastText              32.6 s        0.30 s     16.0 MB     16.0 MB  same
doc2vec (PV-DBOW)               9.4 s        1.81 s     16.0 MB     16.0 MB  same


### Melez yaklaşım

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/04-karsilastirma.html#melez-yaklaşım-ve-bir-olumsuz-sonuç)

Karakter TF-IDF'ini (yazım) ortalaması alınmış fastText ile (anlam) uç uca ekleyin ve
ikisini tek bir sınıflandırıcıya verin. Bölüm önce iki bloğun ölçeğine bakıyor, çünkü tek
bir `C` değeri olan L2 cezalı bir sınıflandırıcı blokları büyüklüklerine göre değerlendiriyor.

In [31]:
dense_train, dense_test = X_train["averaged fastText"], X_test["averaged fastText"]
char_train, char_test = X_train["tf-idf char_wb 3-5"], X_test["tf-idf char_wb 3-5"]
row_norm = np.linalg.norm(dense_train, axis=1)
print("### block scales")
print(f"averaged fastText rows:  mean L2 norm {row_norm.mean():.2f}, mean |component| {np.abs(dense_train).mean():.2f}"
      "      (book: 2.5 and 0.2)")
print(f"char_wb TF-IDF rows:     L2 norm 1.00, non-zeros per row {char_train.nnz / char_train.shape[0]:.0f}, "
      f"mean non-zero value {char_train.data.mean():.2f}   (book: ~301 and 0.05)")

hybrid_train = sp.hstack([char_train, sp.csr_matrix(dense_train)]).tocsr()
hybrid_test = sp.hstack([char_test, sp.csr_matrix(dense_test)]).tocsr()
pred["hybrid"] = new_clf(max_iter=3000).fit(hybrid_train, y_train).predict(hybrid_test)
acc, macro, f1n = scores(y_test, pred["hybrid"])
print(f"\n### tf-idf char_wb 3-5  ⊕  averaged fastText   ({hybrid_train.shape[1]:,} dimensions)")
vs_book("dimensions", hybrid_train.shape[1], 95_039, "{:,}")
vs_book_near("accuracy", acc, 89.36, 0.5)
vs_book_near("macro-F1", macro, 71.65, 0.5)
vs_book_near("F1(neg)", f1n, 49.24, 0.7)
for a, b, book in [("hybrid", "tf-idf char_wb 3-5", "+1.28 [0.65, 1.92]"), ("tf-idf 1-2 gram", "hybrid", "+4.54 [3.42, 5.63]")]:
    observed, low, high = paired_bootstrap(y_test, pred[a], pred[b])
    print(f"{a} − {b}: {observed:+.2f} [{low:.2f}, {high:.2f}]   book {book}")

### block scales
averaged fastText rows:  mean L2 norm 2.53, mean |component| 0.20      (book: 2.5 and 0.2)
char_wb TF-IDF rows:     L2 norm 1.00, non-zeros per row 301, mean non-zero value 0.05   (book: ~301 and 0.05)



### tf-idf char_wb 3-5  ⊕  averaged fastText   (95,039 dimensions)
dimensions                             this run     95,039   book     95,039   same as the book
accuracy                               this run      89.42   book      89.36   +0.06 from the book, inside ±0.5
macro-F1                               this run      71.66   book      71.65   +0.01 from the book, inside ±0.5
F1(neg)                                this run      49.23   book      49.24   -0.01 from the book, inside ±0.7


hybrid − tf-idf char_wb 3-5: +1.28 [0.65, 1.92]   book +1.28 [0.65, 1.92]


tf-idf 1-2 gram − hybrid: +4.54 [3.42, 5.63]   book +4.54 [3.42, 5.63]


**Nasıl okunur.** Düz birleştirme iki parçasının da önünde, kelime bigram TF-IDF'inin ise
gerisinde. Bir melezi yalnızca kendi parçalarıyla karşılaştırmak bir başarı bildirmiş
olurdu.

Bölüm ardından ölçekleri ayarlıyor: yoğun bloğu satır satır L2 ile normalleştir, bir α
katsayısıyla çarp, α ve `C` değerlerini 32.000/8.000 doğrulama bölmesinde seç ve 40.000
satırın tamamında yeniden eğit. Adil bir referans için karakter TF-IDF'i tek başına aynı
`C` ızgarasını alıyor.

**Ağır hücre**: 95.000 sütun üzerinde 24 eğitim, yaklaşık bir dakika. `ANLP_SKIP_HEAVY=1`
altında ızgara, bölümdeki aramanın seçtiği tek nokta.

In [32]:
ALPHAS = [2.0] if SKIP_HEAVY else [0.25, 0.5, 1.0, 2.0, 4.0]
CS = [10.0] if SKIP_HEAVY else [1.0, 3.0, 10.0, 30.0]
fit_i, val_i = train_test_split(positions, test_size=8_000, stratify=y_train, random_state=42)
dense_unit_train, dense_unit_test = normalize(dense_train), normalize(dense_test)


def mix(S, D, alpha):
    return sp.hstack([S, sp.csr_matrix(D * alpha)]).tocsr()


start = time.perf_counter()
char_vec = make_schemes()["tf-idf char_wb 3-5"]
S_fit = char_vec.fit_transform(train["text"].iloc[fit_i])
S_val = char_vec.transform(train["text"].iloc[val_i])
grid = {}
for alpha in ALPHAS:
    for C in CS:
        model = new_clf(C, max_iter=3000).fit(mix(S_fit, dense_unit_train[fit_i], alpha), y_train[fit_i])
        grid[(alpha, C)] = 100 * f1_score(y_train[val_i], model.predict(mix(S_val, dense_unit_train[val_i], alpha)), average="macro")
best_alpha, best_C = max(grid, key=grid.get)
pred["hybrid tuned"] = new_clf(best_C, max_iter=3000).fit(mix(char_train, dense_unit_train, best_alpha), y_train) \
                                                    .predict(mix(char_test, dense_unit_test, best_alpha))
char_grid = {C: 100 * f1_score(y_train[val_i], new_clf(C, max_iter=3000).fit(S_fit, y_train[fit_i]).predict(S_val), average="macro")
             for C in ([30.0] if SKIP_HEAVY else CS)}
best_char_C = max(char_grid, key=char_grid.get)
pred["char tuned"] = new_clf(best_char_C, max_iter=3000).fit(char_train, y_train).predict(char_test)
print(f"### normalised blocks, α and C chosen on validation   ({time.perf_counter() - start:.0f}s)")
note = "grid reduced to one point" if SKIP_HEAVY else None
print("validation macro-F1:  " + "  ".join(f"α={a:g},C={c:g}: {v:.2f}" for (a, c), v in sorted(grid.items(), key=lambda kv_: -kv_[1])[:4]))
vs_book("chosen α", best_alpha, 2.0, note=note)
vs_book("chosen C", best_C, 10.0, note=note)
vs_book_near("tuned hybrid, test macro-F1", scores(y_test, pred["hybrid tuned"])[1], 74.69, 0.5, note=note)
vs_book("char TF-IDF alone, chosen C", best_char_C, 30.0, note=note)
vs_book("char TF-IDF alone, tuned, test macro-F1", scores(y_test, pred["char tuned"])[1], 73.46, "{:.2f}", note=note)
for a, b, book in [("hybrid tuned", "hybrid", "+3.02 [2.00, 4.06]"), ("hybrid tuned", "char tuned", "+1.23 [0.33, 2.15]"),
                   ("tf-idf 1-2 gram", "hybrid tuned", "+1.52 [0.20, 2.80]")]:
    observed, low, high = paired_bootstrap(y_test, pred[a], pred[b])
    print(f"{a} − {b}: {observed:+.2f} [{low:.2f}, {high:.2f}]   book {book}")

### normalised blocks, α and C chosen on validation   (45s)
validation macro-F1:  α=2,C=10: 73.80  α=4,C=10: 73.72  α=0.5,C=30: 73.69  α=1,C=30: 73.66
chosen α                               this run        2.0   book        2.0   same as the book
chosen C                               this run       10.0   book       10.0   same as the book
tuned hybrid, test macro-F1            this run      74.69   book      74.69   -0.00 from the book, inside ±0.5
char TF-IDF alone, chosen C            this run       30.0   book       30.0   same as the book
char TF-IDF alone, tuned, test macro-F1 this run      73.46   book      73.46   same as the book


hybrid tuned − hybrid: +3.02 [2.00, 4.06]   book +3.02 [2.00, 4.06]


hybrid tuned − char tuned: +1.23 [0.33, 2.15]   book +1.23 [0.33, 2.15]


tf-idf 1-2 gram − hybrid tuned: +1.52 [0.20, 2.80]   book +1.52 [0.20, 2.80]


**Nasıl okunur.** Ölçekleri ayarlamak melezin skorunu yaklaşık üç puan değiştiriyor;
dolayısıyla düz birleştirmeyi "melez işe yaramıyor" diye okumak yanlış olurdu. Aynı
ayarlama bütçesi verildiğinde karakter TF-IDF'i tek başına da yükseliyor. Bölümün
koşusunda ayarlanmış melez, *ayarlanmamış* kelime bigram TF-IDF'inin (76,21) gerisinde
kalıyor.

## 12. Unigram torbasının ve ortalamanın ortak sınırı: kelime sırası

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/04-karsilastirma.html#paylaştıkları-başarısızlık)

Aynı kelimelerden oluşan, zıt anlamlı iki yorum. Toplama, terimlerinin sırasına bağlı
değil; bu yüzden unigram TF-IDF de kelime vektörlerinin düz ortalaması da ikisi için aynı
vektörü üretiyor.

**Neye bakacaksınız:** kosinüs, unigram torbası ve ortalama için tam 1; bigram'lar
sayılınca düşüyor. Sonra bigram sınıflandırıcısının bu iki farklı vektörle ne yaptığına
bakın.

In [33]:
review_a, review_b = "kalite iyi fiyat kötü", "kalite kötü fiyat iyi"
print("### two reviews, same words, opposite meaning")
print(f'a = "{review_a}"      (quality good, price bad)')
print(f'b = "{review_b}"      (quality bad, price good)\n')
vs_book("cosine, averaged fastText", cos(doc_vector(review_a, fv), doc_vector(review_b, fv)), 1.0, "{:.4f}")
T = schemes["tf-idf"].transform([review_a, review_b])
vs_book("cosine, tf-idf", cosine_similarity(T[0], T[1])[0, 0], 1.0, "{:.4f}")
T2 = schemes["tf-idf 1-2 gram"].transform([review_a, review_b])
vs_book("cosine, tf-idf 1-2 gram", cosine_similarity(T2[0], T2[1])[0, 0], 0.3247, "{:.4f}")

bigram_model = new_clf().fit(X_train["tf-idf 1-2 gram"], y_train)
bigram_vocab = schemes["tf-idf 1-2 gram"].vocabulary_
print()
for gram in ["kalite iyi", "iyi fiyat", "fiyat kötü", "kalite kötü", "kötü fiyat", "fiyat iyi"]:
    j = bigram_vocab.get(gram)
    print(f"  {gram:<12} " + (f"in the vocabulary, coefficient {bigram_model.coef_[0, j]:+.2f}" if j is not None
                               else "not in the vocabulary (below min_df=2)"))
p_a, p_b = bigram_model.predict_proba(T2)[:, 1]
print()
vs_book("P(positive), review a", p_a, 0.26, "{:.2f}")
vs_book("P(positive), review b", p_b, 0.15, "{:.2f}")

### two reviews, same words, opposite meaning
a = "kalite iyi fiyat kötü"      (quality good, price bad)
b = "kalite kötü fiyat iyi"      (quality bad, price good)

cosine, averaged fastText              this run     1.0000   book     1.0000   same as the book
cosine, tf-idf                         this run     1.0000   book     1.0000   same as the book
cosine, tf-idf 1-2 gram                this run     0.3247   book     0.3247   same as the book

  kalite iyi   in the vocabulary, coefficient -0.62
  iyi fiyat    in the vocabulary, coefficient +0.94
  fiyat kötü   not in the vocabulary (below min_df=2)
  kalite kötü  in the vocabulary, coefficient -0.37
  kötü fiyat   not in the vocabulary (below min_df=2)
  fiyat iyi    in the vocabulary, coefficient -1.05

P(positive), review a                  this run       0.26   book       0.26   same as the book
P(positive), review b                  this run       0.15   book       0.15   same as the book


**Nasıl okunur.** Altı bigram'ın dördü sözlükte, dolayısıyla iki yorum unigram + bigram
TF-IDF'te farklı vektörler alıyor. Farklı vektörler farklı bir etiket, ya da doğru bir
etiket, demek değil: bu model iki yoruma da olumsuz diyor. Bir bigram torbası yalnızca
yerel sırayı tutuyor. Diziyi okuyan modeller
[6. bölümün](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/06-dil-modelleri.html) ve [7. bölümün](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/07-transformerlar.html) konusu.

## 13. Alıştırmalar

Bunlar [4. bölümün kendi alıştırmalarını](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/04-karsilastirma.html#alıştırmalar) yukarıdaki kodla sürdürüyor. Köşeli parantezdeki sayılar
bölümdeki alıştırma numaraları.

1. **Dengesizliği bırakın** [2]. `new_clf` içinde (5. kısım) `class_weight="balanced"`
   ayarını kaldırın ve 5. ile 6. kısımları yeniden çalıştırın. Her satır için doğruluk *ve*
   macro-F1 bildirin. Hangi yöntemler çoğunluk sınıfına çöküyor?
2. **Birleştirici nokta hasarını inceleyin** [7]. Yalnızca U+0307 yüzünden var olan 595
   sütunu 5. kısım saydı. Yalnızca bozuk `i` + U+0307 dizisini onaran bir kural yazın, onu
   `train["text"]` ve `test["text"]` üzerine aynı biçimde uygulayın ve birinci raunt
   tablosunu yeniden çalıştırın. Sözlük boyutundaki değişimi (bölüm 30.195 → 29.823 buldu)
   ve macro-F1'deki değişimi bildirin. Hangisinin daha çok oynayacağını önceden yazın.
3. **Suçlu gerçekten IDF mi?** [10]. Ablasyona (5. kısım) bir basamak ekleyin: her satırı,
   `C` değeri `fit_part` / `val_part` üzerinde seçilmiş olarak tekrarlayın. Normalleştirme
   etkisinin ne kadarı ayakta kalıyor?
4. **Belirsizliği ölçün ve genişletin** [8]. `paired_bootstrap`, `pred` içindeki herhangi
   iki girdi için aralığı veriyor. 7. kısımdaki her çiftin doğruluk farkı için McNemar
   testini ekleyin; 9. kısımdaki gibi köşegen dışı iki sayımı kullanın. Sonra 2. ile 7.
   arasındaki kısımları bölme için beş farklı `random_state` değeriyle tekrarlayın ve her
   farkın ne kadarının bölmeden geldiğini bildirin.
5. **Tekrarlı öğrenme eğrilerini genişletin** [12]. 10. kısımda `N_SAMPLES` değerini 20
   yapın ve `TEN` listesine `"tf-idf char_wb 3-5"` ekleyin. Sonra protokolü değiştirin:
   vektörleştiriciyi yalnızca etiketli örneklemin metninde yeniden eğitin
   (`curve_point` içinde `make_schemes()`) ve eğrinin nasıl değiştiğini bildirin.
6. **Olumsuzlama çiftlerini inceleyin** [4]. 9. kısımdaki `PAIRS` listesini kendi
   alanınızdan yirmi çiftle genişletin. Her model için kosinüsü, tahmin edilen etiketi ve
   `decision_function` değerini kaydedin. Yüksek kosinüse rağmen doğru ayrılan bir çift
   bulun.
7. **SIF ile ağırlıklı ortalama** [3]. `average` işlevini (6. kısım) Arora ve arkadaşlarının
   ağırlıklı ortalamasıyla değiştirin: $a = 10^{-3}$ için $a / (a + p(w))$ ağırlığı,
   `counts` üzerinden kestirilen `p(w)` ve yalnızca eğitim belge vektörlerinden öğrenilen
   ortak bileşen. Düz ortalamayla `paired_bootstrap` kullanarak karşılaştırın.
8. **Bir karar eşiği seçin** [9]. `pred` içinden bir model alın, `val_part` üzerinde
   `predict_proba` kullanın ve üç eşiği değerlendirin. Yanlış alarmları okumak zorunda
   kalacak kişiye hangisini savunacağınıza karar verin ve o eşiğin test karışıklık
   matrisini bir kez bildirin.

Sonraki bölüm bu bölmeyi ve bu modeli koruyup başka bir soru soruyor: etiketler nereden
geldi ve bir kısmı yanlışsa skorlar ne anlama geliyor
([5. bölüm](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/05-veri-etiketleme.html)).

---

*Bu notebook'taki kod MIT lisanslı; anlatım CC BY 4.0. Türkçe ürün yorumları veri kümesi
CC BY-SA 4.0 (Fatih Barmanbay, Hugging Face'te `fthbrmnby/turkish_product_reviews`).*